In [1]:
# رفع ملفات البيانات المطلوبة من الجهاز
from google.colab import files

uploaded = files.upload()
print("✅ الملفات المرفوعة:", list(uploaded.keys()))

Saving all_prompts.json to all_prompts.json
Saving all_prompts_all_folds_combined.csv to all_prompts_all_folds_combined.csv
✅ الملفات المرفوعة: ['all_prompts.json', 'all_prompts_all_folds_combined.csv']


In [ ]:
import os, sys, subprocess

subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run([
    "apt-get", "-qq", "install", "-y",
    "cmake", "libboost-all-dev", "rustc", "cargo"
], check=True)

# لا نلمس numpy/pandas/scipy إذا كانت البيئة جديدة في كولاب.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "camel-tools", "transformers", "sentencepiece",
                       "safetensors", "pyspellchecker", "nltk", "tqdm"])

print("Installation complete. Runtime will restart now...")
os.kill(os.getpid(), 9)

In [1]:
import nltk, subprocess, os, sys
nltk.download("punkt")
nltk.download("stopwords")
subprocess.run(["camel_data", "-i", "defaults"], check=True)
print("NLTK and CAMeL data ready.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


NLTK and CAMeL data ready.


In [2]:

# تأكد سريع من الاستيراد
import numpy as np
import pandas as pd
import camel_tools
import transformers
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("camel_tools imported successfully")

numpy: 1.26.4
pandas: 2.2.2
camel_tools imported successfully


In [3]:
from pathlib import Path
project_dir = Path("/content/aes_project")
project_dir.mkdir(parents=True, exist_ok=True)
print(project_dir)

/content/aes_project


In [4]:
from pathlib import Path

Path("/content/aes_project/camel_tools_init.py").write_text(r'''from camel_tools.disambig.mle import MLEDisambiguator
from camel_tools.tokenizers.word import simple_word_tokenize
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer
from camel_tools.sentiment import SentimentAnalyzer
from camel_tools.tagger.default import DefaultTagger
from transformers import AutoTokenizer, AutoModel
from camel_tools.dialectid import DialectIdentifier
import torch

_mle_disambiguator = None
_morph_analyzer = None
_sentiment_analyzer = None
_bert_tokenizer = None
_bert_model = None
_default_tagger = None
_dialect_id = None
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_disambiguator():
    """Get or initialize the MLE disambiguator."""
    global _mle_disambiguator
    if _mle_disambiguator is None:
        _mle_disambiguator = MLEDisambiguator.pretrained()
    return _mle_disambiguator

def get_analyzer():
    """Get or initialize the morphological analyzer."""
    global _morph_analyzer
    if _morph_analyzer is None:
        # Load the database and create an analyzer instance
        db = MorphologyDB.builtin_db()
        _morph_analyzer = Analyzer(db)
    return _morph_analyzer

# Modified getter functions with error handling
def get_sentiment_analyzer():
    global _sentiment_analyzer
    if _sentiment_analyzer is None:
        _sentiment_analyzer = SentimentAnalyzer.pretrained()
    return _sentiment_analyzer

def get_bert_model(model_name="CAMeL-Lab/bert-base-arabic-camelbert-mix"):
    global _bert_tokenizer, _bert_model
    if _bert_tokenizer is None:
        _bert_tokenizer = AutoTokenizer.from_pretrained(model_name)
    if _bert_model is None:
        _bert_model = AutoModel.from_pretrained(model_name).to(device)
    return _bert_tokenizer, _bert_model

def get_tagger():
    global _default_tagger, _mle_disambiguator
    if _default_tagger is None:
        if _mle_disambiguator is None:
            _mle_disambiguator = get_disambiguator()
        if _mle_disambiguator is not None:
            _default_tagger = DefaultTagger(_mle_disambiguator, 'pos')
    return _default_tagger

def get_dialect_id():
    global _dialect_id
    if _dialect_id is None:
        try:
            # Try with explicit model name
            _dialect_id = DialectIdentifier.pretrained('did-madar-corpus-26')
        except:
            # Fallback to default if that fails
            _dialect_id = DialectIdentifier.pretrained()
    return _dialect_id
''', encoding="utf-8")

print("Wrote camel_tools_init.py")

Wrote camel_tools_init.py


In [5]:
from pathlib import Path

Path("/content/aes_project/essay_proccessing.py").write_text(r'''import re
from nltk.tokenize import word_tokenize
from camel_tools.utils.dediac import dediac_ar
from difflib import SequenceMatcher
from camel_tools.utils.normalize import normalize_unicode


# Alternative approach using findall
def split_into_sentences(essay):
    # Option 2: Using findall to capture sentences with their punctuation
    _SENTENCE_REGEX = re.compile(r'[^.!?;؟،:]*[.!?;؟،:]+')
    sentences = _SENTENCE_REGEX.findall(essay)
    # Handle any remaining text that doesn't end with punctuation
    last_match_end = sum(len(s) for s in sentences)
    if last_match_end < len(essay):
        remaining = essay[last_match_end:].strip()
        if remaining:
            sentences.append(remaining)
    return [sentence.strip() for sentence in sentences if sentence.strip()]

def split_into_paragraphs(essay):
    # this method is just a naive implementation
    # this should be reimplemented
    if not essay or len(essay.strip()) == 0:
        return ["", "", ""]
    essay = essay.strip()
    total_length = len(essay)
    target_length = total_length // 3
    paragraphs = []
    start = 0
    for i in range(2):  # First two paragraphs
        end = start + target_length
        # Try to find a word boundary near the target position
        # Look for space within a reasonable range
        search_range = min(50, target_length // 4)  # Search within 25% of target length
        best_end = end
        for j in range(max(0, end - search_range), min(total_length, end + search_range)):
            if essay[j] == ' ':
                # Prefer positions closer to the target
                if abs(j - end) < abs(best_end - end):
                    best_end = j
        paragraphs.append(essay[start:best_end].strip())
        start = best_end + 1 if best_end < total_length else best_end
    # Add the remaining text as the third paragraph
    paragraphs.append(essay[start:].strip())
    return paragraphs

def split_into_words(essay):
    return word_tokenize(re.sub(r'[^\w\s]', '', essay))


def count_chars(essay):
    char_count = len(essay.replace(" ", ""))
    return char_count


def remove_diatrics_normalizer(essay):
    """
    Remove all Arabic diacritics from the text.

    Args:
        essay (str): Arabic text with diacritics

    Returns:
        str: Text with all diacritics removed
    """
    # Define all Arabic diacritics with their Unicode codes
    diacritics = {
        # Basic short vowels
        '\u064E',  # Fatha (َ)
        '\u064F',  # Damma (ُ)
        '\u0650',  # Kasra (ِ)

        # Tanwin (nunation) - double vowels
        '\u064B',  # Fathatan (ً)
        '\u064C',  # Dammatan (ٌ)
        '\u064D',  # Kasratan (ٍ)

        # Other diacritical marks
        '\u0652',  # Sukoon (ْ)
        '\u0651',  # Shadda (ّ)

        # Additional diacritics
        '\u0653',  # Madda (ٓ)
        '\u0654',  # Hamza above (ٔ)
        '\u0655',  # Hamza below (ٕ)
        '\u0670',  # Superscript alef (ٰ)
    }

    # Remove all diacritics from the text
    cleaned_text = essay
    for diacritic in diacritics:
        cleaned_text = cleaned_text.replace(diacritic, '')

    return cleaned_text

def get_lemmas_and_roots(word_list,_morph_analyzer):
    """Extract lemmas and roots from keywords for robust matching"""
    lemmas = set()
    roots = set()

    for word in word_list:
        # For multi-word phrases, analyze each word
        if ' ' in word:
            words = split_into_words(word)
            for word in words:
                analyses = _morph_analyzer.analyze(word)
                if analyses:
                    # Get lemma and add variations
                    lemma = analyses[0].get('lex', '')
                    if lemma:
                        lemmas.add(dediac_ar(lemma))
                        lemmas.add(lemma)  # Keep original too

                    # Get root if available
                    root = analyses[0].get('root', '')
                    if root:
                        roots.add(dediac_ar(root))
        else:
            # Single word analysis
            analyses = _morph_analyzer.analyze(word)
            if analyses:
                lemma = analyses[0].get('lex', '')
                if lemma:
                    lemmas.add(dediac_ar(lemma))
                    lemmas.add(lemma)

                root = analyses[0].get('root', '')
                if root:
                    roots.add(dediac_ar(root))

            # Also add dediacritized version of original
            lemmas.add(dediac_ar(word))

    return lemmas, roots

def fuzzy_match(text, target_phrase, threshold=0.90):
    """Check if target_phrase appears in text with fuzzy matching"""
    # Normalize both strings
    text = normalize_unicode(text.strip())
    target_phrase = normalize_unicode(target_phrase.strip())

    # Check if target phrase length is reasonable for fuzzy matching
    if len(target_phrase) < 3:
        return target_phrase in text

    # Split text into overlapping windows of similar length to target phrase
    words = text.split()
    target_words = target_phrase.split()
    target_len = len(target_words)

    if target_len == 1:
        # For single word phrases, check each word
        for word in words:
            similarity = SequenceMatcher(None, word, target_phrase).ratio()
            if similarity >= threshold:
                return True
    else:
        # For multi-word phrases, check sliding windows
        for i in range(len(words) - target_len + 1):
            window = ' '.join(words[i:i + target_len])
            similarity = SequenceMatcher(None, window, target_phrase).ratio()
            if similarity >= threshold:
                return True

        # Also check the entire text for longer phrases
        similarity = SequenceMatcher(None, text, target_phrase).ratio()
        if similarity >= threshold:
            return True

    return False
''', encoding="utf-8")

print("Wrote essay_proccessing.py")

Wrote essay_proccessing.py


In [6]:
from pathlib import Path

Path("/content/aes_project/pos_features.py").write_text(r'''from essay_proccessing import split_into_words, split_into_sentences
from collections import Counter
from camel_tools_init import _mle_disambiguator
import numpy as np

def get_all_pos_tags(essays, n=100):
    """
    Get the top n POS tags from a list of essays using CAMeL Tools.
    """
    pos_tags = []
    pos_bigrams = []
    for essay in essays:
        sentences = split_into_sentences(essay)
        for sentence in sentences:
            words = split_into_words(sentence)
            disambiguated = _mle_disambiguator.disambiguate(words)
            sentence_pos_tags = []

            for disambiguated_word in disambiguated:
                if disambiguated_word and len(disambiguated_word) > 0 and disambiguated_word.analyses:
                    analysis = disambiguated_word.analyses[0].analysis
                    if 'pos' in analysis:
                        pos_tags.append(analysis['pos'])
                        sentence_pos_tags.append(analysis['pos'])

            for i in range(len(sentence_pos_tags) - 1):
                bigram = (sentence_pos_tags[i], sentence_pos_tags[i + 1])
                pos_bigrams.append(bigram)

    bigram_counter = Counter(pos_bigrams)
    bigrams_tags = [bigram for bigram, count in bigram_counter.most_common(n)]

    return np.unique(pos_tags), np.unique(bigrams_tags)

def get_top_n_pos_tags(essays, _mle_disambiguator, n=100):
    """
    Get the top n POS tags from a list of essays using CAMeL Tools.
    """
    pos_tags = []
    for essay in essays:
        sentences = split_into_sentences(essay)
        for sentence in sentences:
            words = split_into_words(sentence)
            disambiguated = _mle_disambiguator.disambiguate(words)
            for disambiguated_word in disambiguated:
                if disambiguated_word and len(disambiguated_word) > 0 and disambiguated_word.analyses:
                    analysis = disambiguated_word.analyses[0].analysis
                    if 'pos' in analysis:
                        pos_tags.append(analysis['pos'])

    pos_counter = Counter(pos_tags)
    return [tag for tag, count in pos_counter.most_common(n)]


def get_top_n_pos_bigrams(essays, _mle_disambiguator, n=100):
    """
    Get the top n POS tag bigrams from a list of essays using CAMeL Tools.
    """
    pos_bigrams = []
    for essay in essays:
        sentences = split_into_sentences(essay)
        for sentence in sentences:
            words = split_into_words(sentence)
            disambiguated = _mle_disambiguator.disambiguate(words)
            sentence_pos_tags = []

            # Extract POS tags for this sentence
            for disambiguated_word in disambiguated:
                if disambiguated_word and len(disambiguated_word) > 0 and disambiguated_word.analyses:
                    analysis = disambiguated_word.analyses[0].analysis
                    if 'pos' in analysis:
                        sentence_pos_tags.append(analysis['pos'])

            # Create bigrams from consecutive POS tags in this sentence
            for i in range(len(sentence_pos_tags) - 1):
                bigram = (sentence_pos_tags[i], sentence_pos_tags[i + 1])
                pos_bigrams.append(bigram)

    bigram_counter = Counter(pos_bigrams)
    return [bigram for bigram, count in bigram_counter.most_common(n)]

def get_essay_pos_features(essay, pos_tags_list, pos_bigrams_list, _mle_disambiguator, n=100):
    """
    Get the POS features for an essay using a list of POS tags and POS bigrams.
    Returns counts of tags/bigrams that exist in the provided lists.
    """
    features = {}

    features = {f'pos_{tag}': 0 for tag in pos_tags_list}
    features.update({f'pos_bigram_{b0}_{b1}': 0 for (b0, b1) in pos_bigrams_list})

    all_pos_tags = []
    all_pos_bigrams = []

    sentences = split_into_sentences(essay)
    for sentence in sentences:
        words = split_into_words(sentence)
        disambiguated = _mle_disambiguator.disambiguate(words)
        sentence_pos_tags = []

        # Extract POS tags for this sentence
        for disambiguated_word in disambiguated:
            if disambiguated_word and len(disambiguated_word) > 0 and disambiguated_word.analyses:
                if disambiguated_word.analyses and len(disambiguated_word.analyses) > 0:
                    analysis = disambiguated_word.analyses[0].analysis
                else:
                    analysis = {}
                if 'pos' in analysis:
                    sentence_pos_tags.append(analysis['pos'])

        all_pos_tags.extend(sentence_pos_tags)
        all_pos_bigrams.extend(zip(sentence_pos_tags, sentence_pos_tags[1:]))

    pos_tags_list = set(pos_tags_list)
    pos_bigrams_list = set(pos_bigrams_list)

    tag_counts = Counter(tag for tag in all_pos_tags if tag in pos_tags_list)
    bigram_counts = Counter(b for b in all_pos_bigrams if b in pos_bigrams_list)

    for tag, count in tag_counts.items():
        features[f'pos_{tag}'] = count

    for (b0, b1), count in bigram_counts.items():
        features[f'pos_bigram_{b0}_{b1}'] = count

    return features
''', encoding="utf-8")

print("Wrote pos_features.py")

Wrote pos_features.py


In [7]:
from pathlib import Path

Path("/content/aes_project/readability_measures.py").write_text(r'''import re
from camel_tools.utils.normalize import normalize_unicode
from essay_proccessing import split_into_words, count_chars, split_into_sentences
import math
import re
import pandas as pd
from syntactic_features import calculate_syllable_features, syllabify_arabic_word


def calculate_readability_scores(essay,_mle_disambiguator):
    """
    Calculate readability scores for Arabic text including:
    - FleschReadingEase: adapted for Arabic using custom syllable count
    - SMOGIndex: adapted for Arabic
    - ARI: adapted for Arabic
    - LinsearWrite: adapted for Arabic
    - Kincaid: adapted for Arabic
    - Coleman-Liau: adapted for Arabic
    - LIX: adapted for Arabic
    - RIX: adapted for Arabic
    - GunningFogIndex: adapted for Arabic
    - OSMAN: Arabic Specific Readability Measure
    - AARIBase: Automated Arabic Readability Index
    - Heeti: AlHeeti Grade Level Index
    """
    syllable_features = calculate_syllable_features(essay,_mle_disambiguator)
    syllable_count = syllable_features['syllables']
    complex_words_count = syllable_features['complex_words']
    # Count sentences
    words = split_into_words(essay)
    word_count = len(words)
    char_count = count_chars(essay)
    sentences=split_into_sentences(essay)
    sentence_count = len(sentences)

    long_words_count = 0
    for word in words:
        word_normalized = normalize_unicode(word.strip())
        if len(word_normalized) >= 6:
            long_words_count += 1

    # Calculate percentages and ratios
    words_per_sentence = word_count / sentence_count
    chars_per_word = char_count / word_count
    percent_complex_words = (complex_words_count / word_count) * 100
    percent_long_words = (long_words_count / word_count) * 100

    # Calculate Flesch Reading Ease (adapted for Arabic)
    # Original formula: 206.835 - 1.015 * (words/sentences) - 84.6 * (syllables/words)
    # Adapted weights for Arabic based on research
    flesch_score = 206.835 - (1.015 * (word_count / sentence_count)) - (84.6 * (syllable_count / word_count))
    # Calculate SMOG Index (adapted for Arabic)
    # Original formula: 1.0430 * sqrt(complex_words * (30 / sentence_count)) + 3.1291
    # We'll use the original formula but with our Arabic syllable counting method
    smog_score = 1.0430 * math.sqrt(complex_words_count * (30 / sentence_count)) + 3.1291
    # Calculate Automated Readability Index (ARI)
    # Original: 4.71 * (chars_per_word) + 0.5 * (words_per_sentence) - 21.43
    # Adapted slightly for Arabic
    ari_score = 4.71 * chars_per_word + 0.5 * words_per_sentence - 21.43

    # Calculate Linsear Write Formula
    # Original: (easy_words*1 + difficult_words*3)/sentence_count
    # where easy_words have 1-2 syllables, difficult have 3+ syllables
    easy_words_count = word_count - complex_words_count
    linsear_score = (easy_words_count * 1 + complex_words_count * 3) / sentence_count
    linsear_score = (linsear_score - 2) / 2 if linsear_score <= 20 else linsear_score / 2

    # Calculate Flesch-Kincaid Grade Level (Kincaid)
    # Original: 0.39 * (words_per_sentence) + 11.8 * (syllables_per_word) - 15.59
    # Adapted for Arabic
    syllables_per_word = syllable_count / word_count
    kincaid_score = 0.39 * words_per_sentence + 11.8 * syllables_per_word - 15.59

    # Calculate Coleman-Liau Index
    # Original: 0.0588 * L - 0.296 * S - 15.8
    # where L = letters per 100 words, S = sentences per 100 words
    letters_per_100_words = chars_per_word * 100
    sentences_per_100_words = (sentence_count / word_count) * 100
    coleman_liau_score = 0.0588 * letters_per_100_words - 0.296 * sentences_per_100_words - 15.8

    # Calculate LIX (Läsbarhetsindex)
    # Original: words_per_sentence + (long_words_count / word_count) * 100
    lix_score = words_per_sentence + percent_long_words

    # Calculate RIX (Reading Index)
    # Original: long_words_count / sentence_count
    rix_score = long_words_count / sentence_count if sentence_count > 0 else 0

    # Calculate Gunning Fog Index
    # Original: 0.4 * ((words/sentences) + 100 * (complex_words/words))
    gunning_fog_score = 0.4 * (words_per_sentence + percent_complex_words)

    # Osman = 200.791 - 1.015 * (A/B) - 24.181 * (C/A + D/A + G/A + H/A)
    # Where:
    # A = word_count
    # B = sentence_count
    # C = number of hard words (words > 5 letters)
    # D = total number of syllables
    # G = number of complex words (words with >4 syllables)
    # H = number of Faseeh words (complex words with specific letters or endings)

    hard_words_count = sum(1 for w in words if len(normalize_unicode(w.strip())) > 5)
    complex_words_4plus = 0
    faseeh_count = 0
    faseeh_letters = set(['ء', 'ىء', 'ذ', 'ظ', 'وء'])
    faseeh_endings = ( 'وا', 'ون')
    for w in words:
        wn = normalize_unicode(w.strip())
        syllables = syllabify_arabic_word(wn,_mle_disambiguator)
        if len(syllables) > 4:
            complex_words_4plus += 1
            if any(l in wn for l in faseeh_letters) or wn.endswith(faseeh_endings):
                faseeh_count += 1
    osman = 200.791 - 1.015 * (word_count / sentence_count) - 24.181 * (
        (hard_words_count / word_count) +
        (syllable_count / word_count) +
        (complex_words_4plus / word_count) +
        (faseeh_count / word_count)
    )

    # Calculate AARIBase
    # NOC = number of characters
    # ACW = average characters per word
    # AWS = average words per sentence
    aari_base = (3.28 * char_count) + (1.43 * chars_per_word) + (1.24 * words_per_sentence)

    # Calculate Heeti
    # AWL = average word length (number of characters / number of words)
    heeti = (chars_per_word * 4.414) - 13.468

    return {
        "FleschReadingEase": flesch_score,
        "SMOGIndex": smog_score,
        "ARI": ari_score,
        "LinsearWrite": linsear_score,
        "Kincaid": kincaid_score,
        "Coleman-Liau": coleman_liau_score,
        "LIX": lix_score,
        "RIX": rix_score,
        "GunningFogIndex": gunning_fog_score,
        "OSMAN": osman,
        "AARIBase": aari_base,
        "Heeti": heeti
    }




# df=pd.read_csv('../../../../shared/Arabic_Dataset/cleaned_cqc.csv')
# print(calculate_readability_scores(df['essay'][0]))
''', encoding="utf-8")

print("Wrote readability_measures.py")

Wrote readability_measures.py


In [8]:
from pathlib import Path

Path("/content/aes_project/semantic_features.py").write_text(r'''from nltk.corpus import stopwords
from essay_proccessing import split_into_sentences
import torch
from transformers import pipeline


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_SENTIMENT_PIPELINE = None

def get_sentiment_pipeline():
    global _SENTIMENT_PIPELINE
    if _SENTIMENT_PIPELINE is None:
        _SENTIMENT_PIPELINE = pipeline('sentiment-analysis', model='CAMeL-Lab/bert-base-arabic-camelbert-mix-sentiment')
    return _SENTIMENT_PIPELINE

# Initialize Arabic stopwords
ARABIC_STOPWORDS = set(stopwords.words('arabic'))


def calculate_sentiment_scores(essay):
    """
    Calculates sentiment scores and proportions for Arabic text.
    Returns default values if sentiment analyzer is not available.
    """

    sentiment_analyzer = get_sentiment_pipeline()

    # Normalize and tokenize the text
    sentences = split_into_sentences(essay)
    total_sentences = len(sentences)

    positive_confidence = 0
    negative_confidence = 0
    neutral_confidence = 0

    # Calculate sentiment for each sentence in batches
    batch_size = 8
    positive_scores = []
    negative_scores = []
    neutral_scores = []

    for i in range(0, total_sentences, batch_size):
        batch = sentences[i:i + batch_size]
        sentiments = sentiment_analyzer(batch, truncation=True, max_length=512)

        for sentiment in sentiments:
            if sentiment['label'] == 'positive':
                positive_scores.append(1)
                negative_scores.append(0)
                neutral_scores.append(0)
                positive_confidence += sentiment['score']
            elif sentiment['label'] == 'negative':
                positive_scores.append(0)
                negative_scores.append(1)
                neutral_scores.append(0)
                negative_confidence += sentiment['score']
            else:
                positive_scores.append(0)
                negative_scores.append(0)
                neutral_scores.append(1)
                neutral_confidence += sentiment['score']

    if total_sentences > 0:
        positive_sentence_prop = sum(positive_scores) / total_sentences
        neutral_sentence_prop = sum(negative_scores) / total_sentences
        negative_sentence_prop = sum(neutral_scores) / total_sentences

        overall_positivity = positive_confidence / total_sentences
        overall_negativity = negative_confidence / total_sentences
        overall_neutrality = neutral_confidence / total_sentences
    else:
        positive_sentence_prop = 0.0
        neutral_sentence_prop = 0.0
        negative_sentence_prop = 0.0

        overall_positivity = 0.0
        overall_negativity = 0.0
        overall_neutrality = 0.0

    return {
        "overall_positivity": overall_positivity,
        "overall_negativity": overall_negativity,
        "overall_neutrality": overall_neutrality,
        "positive_sentence_prop": positive_sentence_prop,
        "neutral_sentence_prop": neutral_sentence_prop,
        "negative_sentence_prop": negative_sentence_prop
    }

def calculate_sent_match_words(essay):
    """
    Calculates the number of max and the average number words matched between the sentences in the essay
    """
    sentences = split_into_sentences(essay)
    max_matched_words = 0
    avg_matched_words = 0
    for i in range(len(sentences)):
        for j in range(i + 1, len(sentences)):
            matched_words = len(set(sentences[i].split()) & set(sentences[j].split()))
            max_matched_words = max(max_matched_words, matched_words)
            avg_matched_words += matched_words
    avg_matched_words /= len(sentences) if len(sentences) > 0 else 1
    return {
        "max_matched_words": max_matched_words,
        "avg_matched_words": avg_matched_words
    }

def calculate_prompt_adherence_features(essay, prompt, _bert_tokenizer, _bert_model):
    """
    Calculates prompt adherence features using sentence embeddings with GPU acceleration.
    """

    def get_embedding(texts, batch_size=8):
        embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            batch_inputs = _bert_tokenizer(batch, return_tensors="pt", truncation=True,
                                  max_length=512, padding=True)
            batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}

            with torch.no_grad():
                outputs = _bert_model(**batch_inputs)
            batch_embeddings = outputs.last_hidden_state.mean(dim=1)
            embeddings.extend(batch_embeddings.cpu())
        return embeddings

    prompt_embedding = get_embedding([prompt])
    sentences = split_into_sentences(essay)
    prompt_embedding = prompt_embedding[0].cpu()

    if sentences:
        sentence_embeddings = get_embedding(sentences)
        dot_scores = torch.stack([torch.dot(emb, prompt_embedding) for emb in sentence_embeddings])

        features = {
            "max_sentence_dot_score": dot_scores.max().item(),
            "mean_sentence_dot_score": dot_scores.mean().item(),
            "min_sentence_dot_score": dot_scores.min().item(),
            "dot_score": torch.dot(get_embedding([essay])[0].cpu(), prompt_embedding).item()
        }
    else:
        features = {
            "max_sentence_dot_score": 0.0,
            "mean_sentence_dot_score": 0.0,
            "min_sentence_dot_score": 0.0,
            "dot_score": torch.dot(get_embedding([essay])[0].cpu(), prompt_embedding).item()
        }

    return features

def calculate_sim(text1, text2, _bert_tokenizer, _bert_model):
    """
    Calculates semantic similarity between two texts (paragraphs or sentences) using embeddings.

    Args:
        text1 (str): First text to compare
        text2 (str): Second text to compare

    Returns:
        float: Maximum similarity score between the texts
    """
    def get_embedding(text):
        inputs = _bert_tokenizer(text, return_tensors="pt", truncation=True,
                               max_length=512, padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = _bert_model(**inputs)
        return outputs.last_hidden_state.mean(dim=1)[0].cpu()

    embedding1 = get_embedding(text1)
    embedding2 = get_embedding(text2)

    norm1 = torch.norm(embedding1)
    norm2 = torch.norm(embedding2)
    if norm1 > 0 and norm2 > 0:
        similarity = torch.dot(embedding1, embedding2) / (norm1 * norm2)
    else:
        similarity = torch.tensor(0.0)

    return similarity.item()


def calculate_semantic_similarities(intro, body, conclusion, _bert_tokenizer, _bert_model):
    """
    Calculates semantic similarities between essay parts (intro, body, conclusion) at both paragraph and sentence levels.
    """
    intro_sentences = split_into_sentences(intro)
    body_sentences = split_into_sentences(body)
    conclusion_sentences = split_into_sentences(conclusion)

    paragraphs = [intro, body, conclusion]
    paragraph_similarities = []
    for i in range(len(paragraphs)):
        for j in range(i + 1, len(paragraphs)):
            similarity = calculate_sim(paragraphs[i], paragraphs[j], _bert_tokenizer, _bert_model)
            paragraph_similarities.append(similarity)

    max_paragraph_sim = max(paragraph_similarities) if paragraph_similarities else 0.0
    avg_paragraph_sim = sum(paragraph_similarities) / len(paragraph_similarities) if paragraph_similarities else 0.0

    all_sentences = intro_sentences + body_sentences + conclusion_sentences
    sentence_similarities = []
    if len(all_sentences) > 1:
        for i in range(len(all_sentences)):
            for j in range(i + 1, len(all_sentences)):
                similarity = calculate_sim(all_sentences[i], all_sentences[j], _bert_tokenizer, _bert_model)
                sentence_similarities.append(similarity)

    max_sent_sim = max(sentence_similarities) if sentence_similarities else 0.0
    avg_sent_sim = sum(sentence_similarities) / len(sentence_similarities) if sentence_similarities else 0.0

    return {
        "max_paragraph_sim": max_paragraph_sim,
        "avg_paragraph_sim": avg_paragraph_sim,
        "max_sent_sim": max_sent_sim,
        "avg_sent_sim": avg_sent_sim
    }
''', encoding="utf-8")

print("Wrote semantic_features.py")

Wrote semantic_features.py


In [9]:
from pathlib import Path
Path("/content/aes_project/surface_level_features.py").write_text (r'''from essay_proccessing import split_into_words, split_into_sentences, fuzzy_match
from camel_tools_init import _morph_analyzer
import numpy as np
from nltk.corpus import stopwords
from camel_tools.utils.normalize import normalize_unicode
import math
import re


ARABIC_STOPWORDS = set(stopwords.words('arabic'))



def calculate_lemma_features(essay,_morph_analyzer):
    """
    Calculate lemma-related features for Arabic text using CAMeL Tools morphological analyzer.
    Returns only total unique lemmas and average lemma length.
    """
    # Initialize feature counts
    features = {
        "total_lemmas": 0,        # Total number of unique lemmas
        "avg_lemma_length": 0,    # Average length of lemmas
    }

    # Set to store unique lemmas
    lemma_types = set()

    # Split text into words
    words = split_into_words(essay)

    # Process each word
    for word in words:
        # Get morphological analysis
        analyses = _morph_analyzer.analyze(word)

        if analyses:
            # Get the first analysis (most likely)
            analysis = analyses[0]

            # Extract lemma if available
            if 'lex' in analysis:
                lemma = analysis['lex']
                lemma_types.add(lemma)

    # Calculate final metrics
    features["total_lemmas"] = len(lemma_types)

    # Calculate average lemma length
    if features["total_lemmas"] > 0:
        total_length = sum(len(lemma) for lemma in lemma_types)
        features["avg_lemma_length"] = total_length / features["total_lemmas"]
    return features

def long_words_count(essay):
    """
    Counts the number of words with 7 or more characters in the essay.
    """
    words = split_into_words(essay)
    long_words_count = sum(1 for word in words if len(word) >= 7)
    return long_words_count


def calculate_variance_features(essay):
    """
    Calculates variance-related features for Arabic text including:
    - sent_var: variance of sentence lengths
    - word_var: variance of word lengths
    - stop_prop: proportion of stopwords
    - unique_word: total number of unique words
    - type_token_ratio: ratio of unique words to total words
    """
    # Process words
    words = split_into_words(essay)
    words_count = len(words)

    # Calculate word length variance
    word_lengths = [len(word) for word in words]
    word_var = np.var(word_lengths)

    # Calculate sentence length variance
    sentences = split_into_sentences(essay)
    sentence_lengths = [len(sentence) for sentence in sentences]
    sent_var = np.var(sentence_lengths)

    # Calculate stopword proportion using module-level Arabic stopwords
    stop_words_count = sum(1 for word in words if word in ARABIC_STOPWORDS)
    stop_prop = stop_words_count / words_count if words_count > 0 else 0

    # Calculate unique words and type-token ratio
    unique_words = set(words)
    unique_word_count = len(unique_words)
    type_token_ratio = unique_word_count / words_count if words_count > 0 else 0

    return {
        "sent_var": sent_var,
        "word_var": word_var,
        "stop_prop": stop_prop,
        "type_token_ratio": type_token_ratio
    }

def calculate_punctuation_counts(essay,_mle_disambiguator):
    """
    Counts various punctuation marks in an Arabic essay:
    - Exclamation mark count (!)
    - Semicolon count (;)
    - Double quotes count ("")
    - Dash count (-)
    - Colon count (:)
    - Question mark count (?)
    - Period count (.)
    - Comma count (,)
    - Apostrophe count (')
    - Quotation mark count (")
    - Parenthesis count (()
    - Total punctuation count from morphological analysis

    Args:
        essay (str): The Arabic essay text
        _morph_analyzer: The CAMeL Tools morphological analyzer

    Returns:
        dict: Dictionary containing counts of each punctuation mark
    """
    #how many times the punc tag appears in the essay
    punc_count = 0
    normalized_essay=normalize_unicode(essay)
    disambiguated = _mle_disambiguator.disambiguate(normalized_essay)
    if disambiguated:
        for disambiguated_word in disambiguated:
            if disambiguated_word and len(disambiguated_word) > 0 and disambiguated_word.analyses:
                analysis = disambiguated_word.analyses[0].analysis
                if 'pos' in analysis:
                    if analysis['pos'] == 'punc':
                        punc_count += 1

    return {
        "exclamation_count": essay.count('!'),
        "semicolon_count": essay.count(';') + essay.count('؛'),
        "dash_count": essay.count('-'),
        "colon_count": essay.count(':'),
        "question_count": essay.count('?') + essay.count('؟'),
        "period_count": essay.count('.'),
        "comma_count": essay.count(',') + essay.count('،'),
        "quotation_mark_count": essay.count('"') + essay.count('“') + essay.count('”'),
        "parenthesis_count": essay.count('(') + essay.count(')'),
        "punc_count": punc_count
    }

def calculate_dup_punctuation_count(essay,_mle_disambiguator):
    """"
    check the number of times there is a two consecutive punctuation marks
    """
    normalized_essay=normalize_unicode(essay)
    disambiguated = _mle_disambiguator.disambiguate(normalized_essay)
    dup_punc_count = 0
    prev_was_punc = False

    if disambiguated:
        for disambiguated_word in disambiguated:
            if disambiguated_word and len(disambiguated_word) > 0 and disambiguated_word.analyses:
                analysis = disambiguated_word.analyses[0].analysis
                if 'pos' in analysis:
                    is_punc = analysis['pos'] == 'punc'
                    if is_punc and prev_was_punc:
                        dup_punc_count += 1
                    prev_was_punc = is_punc
                else:
                    prev_was_punc = False
            else:
                prev_was_punc = False

    return dup_punc_count

def calculate_religious_phrases(intro_paragraph,body_paragraph,conclusion_paragraph):
    """
    Checks for religious phrases in specific paragraphs of the essay using fuzzy matching (93% similarity):
    - basmallah_use: First or second paragraph contains بسم الله الرحمن الرحيم
    - hamd_use: First or second paragraph contains الحمد لله رب العالمين
    - amma_baad_use: First or second paragraph contains أما بعد
    - salla_allah_use: First or second paragraph contains صلى الله وبارك
    - sallam_use: Last paragraph contains والسلام عليكم ورحمة الله وبركاته
    - salla_alla_mohammed_use: Last paragraph contains وصلى الله وسلم على نبينا محمد

    Args:
        intro_paragraph (str): The introduction paragraph
        body_paragraph (str): The body paragraph
        conclusion_paragraph (str): The conclusion paragraph

    Returns:
        dict: Dictionary containing boolean values for each religious phrase check
    """


    # Split essay into paragraphs
    paragraphs = [intro_paragraph, body_paragraph, conclusion_paragraph]

    # Initialize result dictionary
    result = {
        "basmallah_use": 0,
        "hamd_use": 0,
        "amma_baad_use": 0,
        "salla_allah_use": 0,
        "sallam_use": 0,
        "salla_alla_mohammed_use": False
    }

    # Religious phrases to search for
    opening_phrases = {
        "basmallah_use": "بسم الله",
        "hamd_use": "الحمد لله",
        "amma_baad_use": "أما بعد",
        "salla_allah_use": "صلى الله"
    }

    closing_phrases = {
        "sallam_use": "والسلام عليكم",
        "salla_alla_mohammed_use": "وصلى الله وسلم على نبينا محمد"
    }

    # Check first and second paragraphs for opening phrases
    for i in range(min(2, len(paragraphs))):
        paragraph = paragraphs[i]
        for key, phrase in opening_phrases.items():
            if fuzzy_match(paragraph, phrase,0.93):
                result[key] = 1

    # Check last paragraph for closing phrases
    if paragraphs:
        last_paragraph = paragraphs[-1]
        for key, phrase in closing_phrases.items():
            if fuzzy_match(last_paragraph, phrase,0.93):
                result[key] = 1

    return result

def _has_causative_prefix(word, _mle_disambiguator, causative_prefixes):
    """
    Check if a word has a causative prefix using CAMeL Tools morphological analysis.

    Args:
        word (str): The Arabic word to analyze
        _mle_disambiguator: The CAMeL Tools disambiguator
        causative_prefixes (list): List of potential causative prefixes

    Returns:
        bool: True if the word has a causative prefix, False otherwise
    """
    try:
        # Normalize and disambiguate the word
        normalized_word = normalize_unicode(word)
        disambiguated = _mle_disambiguator.disambiguate(normalized_word)

        if disambiguated and len(disambiguated) > 0:
            for disambiguated_word in disambiguated:
                if (disambiguated_word and len(disambiguated_word) > 0 and
                    disambiguated_word.analyses):
                    analysis = disambiguated_word.analyses[0].analysis

                    # Check if there's a prefix in the analysis
                    if 'prc0' in analysis or 'prc1' in analysis or 'prc2' in analysis or 'prc3' in analysis:
                        # Get all prefixes
                        prefixes = []
                        for i in range(4):  # prc0, prc1, prc2, prc3
                            prc_key = f'prc{i}'
                            if prc_key in analysis and analysis[prc_key]:
                                prefixes.append(analysis[prc_key])

                        # Check if any of the prefixes match our causative prefixes
                        for prefix in prefixes:
                            if prefix in causative_prefixes:
                                return True

                    # Also check the base word (bw) to see if the prefix is separated
                    if 'bw' in analysis:
                        base_word = analysis['bw']
                        # If the original word is longer than base word + known prefixes,
                        # it might have a causative prefix
                        for causative_prefix in causative_prefixes:
                            if word.startswith(causative_prefix) and base_word:
                                # Check if removing the prefix gives us something close to the base word
                                word_without_prefix = word[len(causative_prefix):]
                                if word_without_prefix and len(word_without_prefix) >= len(base_word) - 2:
                                    return True

        return False
    except Exception:
        # Fallback to simple prefix check if analysis fails
        return any(word.startswith(prefix) for prefix in causative_prefixes)

def calculate_advanced_punctuation_features(essay, _mle_disambiguator):
    """
    Analyzes advanced punctuation usage in Arabic text according to specific rules.

    Args:
        essay (str): The Arabic essay text
        _mle_disambiguator: The CAMeL Tools morphological analyzer

    Returns:
        dict: Dictionary containing counts of correct, missing, and incorrect uses of various punctuation marks
    """
    # Initialize counters
    features = {
        # Question mark features
        "question_mark_correct": 0,  # Correct question mark usage
        "question_mark_missing": 0,  # Missing question mark
        "question_mark_incorrect": 0,  # Incorrect question mark usage

        # Exclamation mark features
        "exclamation_mark_correct": 0,  # Correct exclamation mark usage
        "exclamation_mark_missing": 0,  # Missing exclamation mark
        "exclamation_mark_incorrect": 0,  # Incorrect exclamation mark usage

        # Semicolon features
        "semicolon_correct": 0,  # Correct semicolon usage
        "semicolon_missing": 0,  # Missing semicolon
        "semicolon_incorrect": 0,  # Incorrect semicolon usage

        # Comma features
        "comma_incorrect": 0,  # Incorrect comma usage with discourse connectives
        "comma_missing": 0,  # Missing comma with discourse connectives

        # Period features
        "period_correct": 0,  # Correct period usage
        "period_incorrect": 0,  # Incorrect period usage

        # Colon features
        "colon_correct": 0,  # Correct colon usage
        "colon_missing": 0,  # Missing colon
        "colon_incorrect": 0,  # Incorrect colon usage

        # Quotation mark features
        "quotation_mark_correct": 0,  # Correct quotation mark usage
        "quotation_mark_missing": 0,  # Missing quotation mark
        "quotation_mark_incorrect": 0,  # Incorrect quotation mark usage
    }

    # Question tools
    question_tools = ['هل', 'كيف', 'ماذا', 'لماذا', 'لم', 'كم', 'متى', 'أين']
    question_tools_lemmas = []
    for tool in question_tools:
        try:
            normalized_tool = normalize_unicode(tool)
            disambiguated = _mle_disambiguator.disambiguate(normalized_tool)
            if disambiguated and len(disambiguated) > 0:
                for disambiguated_word in disambiguated:
                    if (disambiguated_word and len(disambiguated_word) > 0 and
                        disambiguated_word.analyses and len(disambiguated_word.analyses) > 0):
                        analysis = disambiguated_word.analyses[0].analysis
                        if 'lex' in analysis:
                            question_tools_lemmas.append(analysis['lex'])
                            break
        except (IndexError, AttributeError, KeyError):
            # If analysis fails, use the original tool
            question_tools_lemmas.append(tool)

    # Exaggerating styles
    exaggerating_styles = ['ياليت', 'بئس', 'رائع', 'لله در']

    # Causative indicators
    causative_indicators = ['لأن', 'بسبب', 'لكي']
    causative_prefixes = ['ل', 'ف']

    # Colon indicators
    colon_indicators = ['مثال', 'التالية', 'الآتية', 'مايلي','قال']

    # Split into sentences and paragraphs
    sentences = split_into_sentences(essay)
    paragraphs = essay.split('\n\n')

    # Process each sentence
    for sentence in sentences:
        words = split_into_words(sentence)

        # Extract lemmas from words with proper error handling
        lemmas = []
        for word in words:
            try:
                normalized_word = normalize_unicode(word)
                disambiguated = _mle_disambiguator.disambiguate(normalized_word)
                if disambiguated and len(disambiguated) > 0:
                    for disambiguated_word in disambiguated:
                        if (disambiguated_word and len(disambiguated_word) > 0 and
                            disambiguated_word.analyses and len(disambiguated_word.analyses) > 0):
                            analysis = disambiguated_word.analyses[0].analysis
                            if 'lex' in analysis:
                                lemmas.append(analysis['lex'])
                                break
            except (IndexError, AttributeError, KeyError):
                # Skip words that can't be analyzed
                continue

        # Question mark analysis
        # has_question_tool = any(fuzzy_match(sentence, tool, 0.99) for tool in question_tools)
        # Exact match for question mark between words and question_tools
        has_question_tool = any([tool==word for word in lemmas for tool in question_tools_lemmas])
        has_question_mark = '?' in sentence or '؟' in sentence

        if has_question_tool and has_question_mark:
            features["question_mark_correct"] += 1
        elif has_question_tool and not has_question_mark:
            features["question_mark_missing"] += 1
        elif not has_question_tool and has_question_mark:
            features["question_mark_incorrect"] += 1

        # Exclamation mark analysis
        has_exaggerating_style = any(fuzzy_match(sentence, style, 0.95) for style in exaggerating_styles)
        has_exclamation_mark = '!' in sentence

        if has_exaggerating_style and has_exclamation_mark:
            features["exclamation_mark_correct"] += 1
        elif has_exaggerating_style and not has_exclamation_mark:
            features["exclamation_mark_missing"] += 1
        elif not has_exaggerating_style and has_exclamation_mark:
            features["exclamation_mark_incorrect"] += 1

        # Semicolon analysis
        if ';' in sentence or '؛' in sentence:
            next_word = None
            for i, word in enumerate(words):
                if (word == ';' or word == '؛') and i + 1 < len(words):
                    next_word = words[i + 1]
                    break

            if next_word:
                has_causative = (any(fuzzy_match(next_word, indicator, 0.95) for indicator in causative_indicators) or
                               _has_causative_prefix(next_word, _mle_disambiguator, causative_prefixes))
                if has_causative:
                    features["semicolon_correct"] += 1
                else:
                    features["semicolon_incorrect"] += 1
        else:
            # Check for missing semicolon
            for i, word in enumerate(words):
                if (any(fuzzy_match(word, indicator, 0.95) for indicator in causative_indicators) or
                    _has_causative_prefix(word, _mle_disambiguator, causative_prefixes)):
                    if i > 0 and words[i-1] != ';' and words[i-1] != '؛':
                        features["semicolon_missing"] += 1

        # Colon analysis
        has_colon_indicator = any(fuzzy_match(sentence, indicator, 0.95) for indicator in colon_indicators)
        has_colon = ':' in sentence

        if has_colon_indicator and has_colon:
            features["colon_correct"] += 1
        elif has_colon_indicator and not has_colon:
            features["colon_missing"] += 1
        elif not has_colon_indicator and has_colon:
            features["colon_incorrect"] += 1

    # Process paragraphs for period and comma analysis
    for paragraph in paragraphs:
        # Period analysis
        if paragraph.strip().endswith('.'):
            features["period_correct"] += 1
        elif '.' in paragraph[:-1]:  # Period before end
            features["period_incorrect"] += 1

        # Comma analysis with discourse connectives
        # Note: This is a simplified version. You may want to add more discourse connectives
        discourse_connectives = {
        # Contrast and Opposition
        'الا ان': ['الا ان', 'الا أن'],
        'بيد ان': ['بيد ان', 'بيد أن'],
        'غير ان': ['غير ان', 'غير أن'],
        'على الرغم': ['على الرغم'],
        'رغمان': ['رغمان'],
        'بالرغم من': ['بالرغم من'],
        'برغم': ['برغم'],
        'بالمقابل': ['بالمقابل'],
        'في المقابل': ['في المقابل'],
        'بيد': ['بيد'],

        # Time and Sequence
        'بعدما': ['بعدما'],
        'اذ': ['اذ', 'إذ'],
        'بينما': ['بينما'],
        'عقب': ['عقب'],
        'قبيل': ['قبيل'],
        'وقبل': ['وقبل'],
        'من ثم': ['من ثم'],
        'قبل ان': ['قبل ان', 'قبل أن'],

        # Cause and Effect
        'جراء': ['جراء'],
        'نظرا ل': ['نظرا ل'],
        'بفضل': ['بفضل'],
        'لأن': ['لأن'],
        'بحيث': ['بحيث'],

        # Condition and Exception
        'الا اذا': ['الا اذا', 'الا إذا'],
        'حتى لو': ['حتى لو'],
        'لولا': ['لولا'],
        'طالما': ['طالما'],
        'كلما': ['كلما'],

        # Purpose and Comparison
        'بغية': ['بغية'],
        'كأن': ['كأن'],
        'خلافا ل': ['خلافا ل'],
        'بمعنى اخر': ['بمعنى اخر', 'بمعنى آخر'],

        # Context and Situation
        'في ظل': ['في ظل'],
        'حال': ['حال']
    }

        for connective in discourse_connectives:
            if fuzzy_match(paragraph, connective, 0.95):
                if ',' not in paragraph:
                    features["comma_incorrect"] += 1
                elif ',' in paragraph and not any(fuzzy_match(word, indicator, 0.95) for word in split_into_words(paragraph) for indicator in causative_indicators):
                    features["comma_missing"] += 1

    # Quotation mark analysis
    # Comprehensive list of attribution cues categorized by type
    attribution_cues = {
        "assertion": [  # Declarative/assertive reporting
            "قال",      # said
            "ذكر",      # mentioned
            "صرح",      # declared
            "أعلن",     # announced
            "أفاد",     # stated
            "أكد",      # confirmed
            "جزم"       # asserted
        ],
        "directive": [  # Requesting/questioning
            "سأل",      # asked
            "طلب",      # requested
            "أمر",      # ordered
            "استفهم"    # questioned/inquired
        ],
        "expression": [  # Emotions, social acts
            "اعتذر",    # apologized
            "شكر",      # thanked
            "هنأ"       # congratulated
        ],
        "commissive": [  # Commitments/promises
            "وعد",      # promised
            "أقسم",     # swore
            "راهن"      # bet
        ],
        "declarative": [  # State-changing speech acts
            "أبلغ",     # informed
            "اعترف"     # admitted
        ],
        "adverbs_adjectives": [  # Used as modifiers in implicit attributions
            "مضيفاً",   # adding
            "معلقاً",   # commenting
            "مؤكداً",   # emphasizing
            "واصفاً"    # describing
        ],
        "prepositional_phrases": [  # Used with indirect attributions
            "بحسب",     # according to
            "وفقاً لـ",  # according to
            "على حد قوله"  # as he said / according to his words
        ]
    }

    # Flatten the attribution cues dictionary into a single list for easier checking
    all_attribution_cues = []
    for category in attribution_cues.values():
        all_attribution_cues.extend(category)

    for sentence in sentences:
        # Check for explicit attribution cues
        has_attribution = any(fuzzy_match(sentence, cue, 0.95) for cue in all_attribution_cues)

        # Check for implicit patterns (name followed by colon)
        words = split_into_words(sentence)
        for i, word in enumerate(words):
            if i + 1 < len(words) and words[i + 1] == ':':
                has_attribution = True
                break

        has_quotes = '"' in sentence

        if has_attribution and has_quotes:
            features["quotation_mark_correct"] += 1
        elif has_attribution and not has_quotes:
            features["quotation_mark_missing"] += 1
        elif not has_attribution and has_quotes:
            features["quotation_mark_incorrect"] += 1

    return features


def extract_surface_features(essay,intro_paragraph,body_paragraph,conclusion_paragraph):
    #words
    if essay.strip() != '':
        words = split_into_words(essay)
        words_count = len(words)
        log_words_count = math.log10(words_count) if words_count > 0 else 0
        unique_words = set(words)
        unique_words_count = len(unique_words) #the set() removes duplicates
        log_unique_words_count = math.log10(unique_words_count) if unique_words_count > 0 else 0
        total_word_length = sum(len(word) for word in unique_words)#for unique words
        average_word_length = total_word_length / unique_words_count if unique_words_count > 0 else 0
        max_length_word = max(len(word) for word in unique_words) if unique_words else 0
        min_length_word = min(len(word) for word in unique_words) if unique_words else 0
        squared_diffs_words = [(len(word) - average_word_length) ** 2 for word in unique_words]
        mean_squared_diffs_words = sum(squared_diffs_words) / len(squared_diffs_words) if len(squared_diffs_words) > 0 else 0
        standard_deviation_words= math.sqrt(mean_squared_diffs_words) if mean_squared_diffs_words > 0 else 0 #the standard deviation as a way to understand how much individual values within a group differ from the average value of that group

        #General counts
        chars_count = len(essay.replace(" ", "")) #not counting spaces
        hmpz_count = len(re.findall(r'[أإءؤئ]', essay))# Number of <hmzp> (F22)
    else:
        words_count = 0
        log_words_count = 0
        unique_words_count = 0
        log_unique_words_count = 0
        average_word_length = 0
        max_length_word = 0
        min_length_word = 0
        standard_deviation_words = 0
        chars_count = 0
        hmpz_count = 0


    if intro_paragraph.strip() != '' or body_paragraph.strip() != '' or conclusion_paragraph.strip() != '':
        #Paragraphs
        paragraphs = [p for p in [intro_paragraph, body_paragraph, conclusion_paragraph] if p.strip() != '']
        paragraphs_count =len(paragraphs)  #num_paragraphs = len(essay.split('\n')) #Number of paragraphs (F3)
        is_first_paragraph_less_than_or_equal_to_10 = int(len(split_into_words(paragraphs[0])) <= 10) if paragraphs else 0 #(F16)
        paragraphs_lengths = [len(split_into_words(paragraph)) for paragraph in paragraphs] #length of each paragraph interms of words
        average_length_paragraph = sum(paragraphs_lengths)/ paragraphs_count if paragraphs_count > 0 else 0 # Average length of paragraph (F11)
        max_length_paragraph = max(paragraphs_lengths) if paragraphs_lengths else 0 # Maximum length of paragraph (F12)
        min_length_paragraph = min(paragraphs_lengths) if paragraphs_lengths else 0 # Minimum length of paragraph (F13)
    if essay.strip() != '':
        #Sentences
        sentences = split_into_sentences(essay)
        sentences_count = len(sentences) # Number of sentences (F5)
        sentence_lengths = [len(split_into_words(sentence)) for sentence in sentences]
        average_length_sentence = sum(sentence_lengths) / sentences_count if sentences_count > 0 else 0    # Average length of sentence (F10)
        max_length_sentence = max(sentence_lengths) if sentence_lengths else 0 # Maximum length of sentence
        min_length_sentence = min(sentence_lengths) if sentence_lengths else 0 # Minimum length of sentence
        squared_diff_sentence = [(length - average_length_sentence) ** 2 for length in sentence_lengths]
        mean_squared_diff_sentence = np.mean(squared_diff_sentence) if len(squared_diff_sentence) > 0 else 0
        standard_deviation_sentence = np.sqrt(mean_squared_diff_sentence) if mean_squared_diff_sentence > 0 else 0
    else:
        sentences_count = 0
        average_length_sentence = 0
        max_length_sentence = 0
        min_length_sentence = 0
        standard_deviation_sentence = 0


    #Grouping the features into a list

    if intro_paragraph.strip() != '' or body_paragraph.strip() != '' or conclusion_paragraph.strip() != '':
        extracted_surface_features= [words_count,log_words_count,unique_words_count,log_unique_words_count,
        average_word_length,max_length_word,min_length_word,standard_deviation_words,chars_count,hmpz_count,
        paragraphs_count,is_first_paragraph_less_than_or_equal_to_10,average_length_paragraph,
        max_length_paragraph, min_length_paragraph, sentences_count, average_length_sentence,
        max_length_sentence, min_length_sentence,standard_deviation_sentence]
        features = {
            "words_count": words_count,
            "log_words_count": log_words_count,
            "unique_words_count": unique_words_count,
            "log_unique_words_count": log_unique_words_count,
            "average_word_length": average_word_length,
            "max_length_word": max_length_word,
            "min_length_word": min_length_word,
            "standard_deviation_words": standard_deviation_words,
            "chars_count": chars_count,
            "hmpz_count": hmpz_count,
            "paragraphs_count": paragraphs_count,
            "is_first_paragraph_less_than_or_equal_to_10": is_first_paragraph_less_than_or_equal_to_10,
            "average_length_paragraph": average_length_paragraph,
            "max_length_paragraph": max_length_paragraph,
            "min_length_paragraph": min_length_paragraph,
            "sentences_count": sentences_count,
            "average_length_sentence": average_length_sentence,
            "max_length_sentence": max_length_sentence,
            "min_length_sentence": min_length_sentence,
            "standard_deviation_sentence": standard_deviation_sentence
        }
    else:
        extracted_surface_features= [words_count,log_words_count,unique_words_count,log_unique_words_count,
        average_word_length,max_length_word,min_length_word,standard_deviation_words,chars_count,hmpz_count,
        sentences_count, average_length_sentence,max_length_sentence, min_length_sentence,standard_deviation_sentence]
        features = {
            "words_count": words_count,
            "log_words_count": log_words_count,
            "unique_words_count": unique_words_count,
            "log_unique_words_count": log_unique_words_count,
            "average_word_length": average_word_length,
            "max_length_word": max_length_word,
            "min_length_word": min_length_word,
            "standard_deviation_words": standard_deviation_words,
            "chars_count": chars_count,
            "hmpz_count": hmpz_count,
            "sentences_count": sentences_count,
            "average_length_sentence": average_length_sentence,
            "max_length_sentence": max_length_sentence,
            "min_length_sentence": min_length_sentence,
            "standard_deviation_sentence": standard_deviation_sentence
        }

    return features











''', encoding="utf-8")
print("Wrote surface_level_features.py")

Wrote surface_level_features.py


In [10]:
from pathlib import Path
Path("/content/aes_project/syntactic_features.py").write_text(r'''from nltk.corpus import stopwords
from camel_tools_init import _mle_disambiguator, _morph_analyzer, _dialect_id
from camel_tools.utils.normalize import normalize_unicode
from essay_proccessing import split_into_words, split_into_sentences, get_lemmas_and_roots
import re
from collections import defaultdict
import torch
from collections import Counter
import subprocess
import os
from camel_tools.utils.dediac import dediac_ar
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Initialize Arabic stopwords
ARABIC_STOPWORDS = set(stopwords.words('arabic'))
from spellchecker import SpellChecker

def syllabify_arabic_word(word,_mle_disambiguator):
    """
    Syllabify an Arabic word based on the rules from Zeki et al.

    Rules:
    - Every syllable begins with a consonant followed by a vowel
    - Syllable types: CV, CVV, CVC, CVVC, CVCC
    """
    # Define Arabic character sets
    consonants = "ةءأؤإئابتثجحخدذرزسشصضطظعغفقكلمنهويى"
    long_vowels = "اوي"  # alif, waw, yaa
    short_vowels = "َُِ"  # fatha, damma, kasra
    sukoon = "ْ"  # sukoon marks absence of vowel
    shadda = "ّ"  # gemination mark (doubles consonant)

    if _mle_disambiguator.disambiguate([word])[0].analyses and len(_mle_disambiguator.disambiguate([word])[0].analyses) > 0:
        diacritized_word = _mle_disambiguator.disambiguate([word])[0].analyses[0].analysis['diac']
    else:
        diacritized_word = word
    word = diacritized_word

    # If last character is a dicaritic, remove it and replace with sukoon
    if word and word[-1] not in consonants:
        word = word[:-1] + sukoon
    elif word and word[-1] != sukoon:
        word += sukoon

    # Normalize word: handle shadda by duplicating the consonant
    normalized = ""
    i = 0
    while i < len(word):
        normalized += word[i]
        if i < len(word) - 1 and word[i+1] == shadda:
            normalized += sukoon
            normalized += word[i]  # Duplicate the consonant
            i += 2
        else:
            i += 1

    word = normalized

    # Syllabify
    syllables = []
    i = 0
    current_syllable = ""

    while i < len(word):
        # Skip non-Arabic characters
        if not (word[i] in consonants or word[i] in short_vowels or
                word[i] in long_vowels or word[i] in sukoon):
            i += 1
            continue

        # 1. Each syllable must start with a consonant
        if word[i] in consonants:
            current_syllable = word[i]
            i += 1

            # 2. Followed by a vowel (short or long)
            if i < len(word):
                # Case: short vowel (diacritic)
                if word[i] in short_vowels:
                    current_syllable += word[i]
                    i += 1

                    # Check for CV pattern (end of syllable)
                    if i >= len(word) or word[i] in consonants:
                        if i < len(word) and word[i] in consonants:
                            # Check for CVCC pattern
                            if i + 3 < len(word) and word[i+1] == sukoon and word[i+2] in consonants and word[i+3] == sukoon:
                                current_syllable += word[i] + word[i+1] + word[i+2] + word[i+3]
                                i += 4
                            # Check for CVC pattern (when followed by two consonants with sukoon)
                            elif i + 1 < len(word) and word[i+1] == sukoon:
                                current_syllable += word[i] + word[i+1]
                                i += 2
                            # elif i + 4 < len(word) and word[i+1] == sukoon and word[i+2] in consonants and word[i+3] == sukoon:
                            #     current_syllable += word[i] + word[i+1] + word[i+2] + word[i+3]
                            #     i += 4
                                # i += 3

                        syllables.append(current_syllable)
                        current_syllable = ""

                # Case: long vowel
                elif word[i] in long_vowels:
                    current_syllable += word[i]
                    i += 1

                    # Check for CVV pattern (end of syllable)
                    if i >= len(word) or word[i] in consonants:
                        if i < len(word) and word[i] in consonants:
                            # Check for CVVC pattern
                            if i + 1 < len(word) and word[i+1] == sukoon:
                                current_syllable += word[i] + word[i+1]
                                i += 2

                        syllables.append(current_syllable)
                        current_syllable = ""
                else:
                    # Consonant without vowel - assume implicit vowel
                    syllables.append(current_syllable)
                    current_syllable = ""
            else:
                # End of word with just a consonant - assume implicit vowel
                syllables.append(current_syllable)
                current_syllable = ""
        else:
            # Skip anything that doesn't start a valid syllable
            i += 1

    # Add any remaining syllable
    if current_syllable:
        syllables.append(current_syllable)

    return syllables


def syllabify_arabic_text(essay,_mle_disambiguator):
    """Syllabify Arabic text with or without diacritics."""
    words = split_into_words(essay)
    all_syllables = []
    for word in words:
        all_syllables.extend(syllabify_arabic_word(word,_mle_disambiguator))
    return all_syllables, words

def calculate_syllable_features(essay,_mle_disambiguator):
    """
    Calculate syllable-related features for Arabic text including:
    - syllables: total number of syllables
    - syll_per_word: average syllables per word
    - complex_words: words with 3+ syllables
    - complex_words_dc: words that would be considered difficult per Dale-Chall criteria
    """
    # Get syllables and words
    syllables, words = syllabify_arabic_text(essay,_mle_disambiguator)
    # Count total syllables
    syllable_count = len(syllables)
    # Get word count
    word_count = len(words)
    # Calculate syllables per word
    syll_per_word = syllable_count / word_count if word_count > 0 else 0
    # Count words that are not in our Arabic adaptation of Dale-Chall list
    complex_words_count = 0

    for i, word in enumerate(words):
        word_normalized = normalize_unicode(word.strip())
        word_syllables = syllabify_arabic_word(word,_mle_disambiguator)
        syllable_count_per_word = len(word_syllables)
        # Count words that would be considered difficult per Dale-Chall criteria
        # For Arabic: Words that are not stopwords AND (have 3+ syllables OR are 6+ characters)
        is_difficult = (syllable_count_per_word >= 3)
        if is_difficult:
            complex_words_count += 1

    return {
        "syllables": syllable_count,
        "syll_per_word": syll_per_word,
        "complex_words": complex_words_count
    }


def calculate_pronoun_features(essay,_mle_disambiguator):
    """Extract pronoun features using CAMeL Tools with optimized processing."""
    features = {}
    pronoun_counts = defaultdict(int)
    group_counts = defaultdict(int)
    # Get cached sentences
    sentences = split_into_sentences(essay)

    # Track sentence-level statistics
    sentences_with_pronoun = defaultdict(set)

    # Process each sentence
    for sent_idx, sentence in enumerate(sentences):

        # Get POS tags using CAMeL Tools
        words = split_into_words(sentence)
        analyses = _mle_disambiguator.disambiguate(words)

        # Process each word analysis
        for word_idx, analysis in enumerate(analyses):
            # Get the top analysis
            if analysis and len(analysis.analyses) > 0:
                top_analysis = analysis.analyses[0].analysis
            else:
                top_analysis = {}
            pos = top_analysis.get('pos', '')
            # Check for pronouns using CAMeL Tools POS tags
            # if pos == 'pron' or pos == 'pron_dem':
            if 'pron' in pos:
                # Get more specific features
                pron_type = pos
                # gen = top_analysis.get('gen', '')
                # num = top_analysis.get('num', '')
                per = top_analysis.get('per', '')
                feature_key = ""
                # # Create pronoun feature name
                # feature_key = f"{pron_type}"
                if per and per != 'na':
                    feature_key += f"_{per}"
                # if gen and gen != 'na':
                #     feature_key += f"_{gen}"
                # if num and num != 'na':
                #     feature_key += f"_{num}"

                feature_key = f"{pron_type}_" + dediac_ar(top_analysis.get('stem', ''))

                # Increment counts
                pronoun_counts[feature_key] += 1
                sentences_with_pronoun[feature_key].add(sent_idx)
                # sentences_with_pronoun[pron_type].add(sent_idx)

                # Count pronoun groups efficiently
                if pos == 'pron_dem':
                    group_counts['demonstrative'] += 1
                    sentences_with_pronoun['demonstrative'].add(sent_idx)
                # elif pos == 'pron' and 'rat' in top_analysis and top_analysis['rat'] == 'r':
                elif pos == 'pron_rel':
                    group_counts['relative'] += 1
                    sentences_with_pronoun['relative'].add(sent_idx)
                elif pos == 'pron_interrog':
                    group_counts['interrogative'] += 1
                    sentences_with_pronoun['interrogative'].add(sent_idx)
                elif pos == 'pron_exclam':
                    group_counts['exclamation'] += 1
                    sentences_with_pronoun['exclamation'].add(sent_idx)
                elif per == '1':
                    group_counts['first_person'] += 1
                    sentences_with_pronoun['first_person'].add(sent_idx)
                elif per == '2':
                    group_counts['second_person'] += 1
                    sentences_with_pronoun['second_person'].add(sent_idx)
                elif per == '3':
                    group_counts['third_person'] += 1
                    sentences_with_pronoun['third_person'].add(sent_idx)

    # Add counts to features (only for pronouns that actually appear)
    for pron_type, count in pronoun_counts.items():
        features[f"pron_{pron_type.lower()}"] = count
        # The number of sentences that contain pron_type
        features[f"sent_count_{pron_type.lower()}"] = len(sentences_with_pronoun[pron_type])
        # The percentage of sentences that contain pronoun pron_type
        features[f"sent_percent_{pron_type.lower()}"] = (len(sentences_with_pronoun[pron_type]) / len(sentences) * 100) if len(sentences) > 0 else 0


    for group, count in group_counts.items():
        features[f"group_{group}"] = count
        # The number of sentences that contain group
        features[f"sent_count_{group}"] = len(sentences_with_pronoun[group])
        # The percentage of sentences that contain group
        features[f"sent_percent_{group}"] = (len(sentences_with_pronoun[group]) / len(sentences) * 100) if len(sentences) > 0 else 0

    # Add sentence-level statistics
    # total_sentences = len(sentences)
    # if total_sentences > 0:
    #     for pron_type, sent_set in sentences_with_pronoun.items():
    #         features[f"sent_count_{pron_type.lower()}"] = len(sent_set)
    #         features[f"sent_percent_{pron_type.lower()}"] = (len(sent_set) / total_sentences) * 100

    return features

def calculate_possessive_features(essay,_morph_analyzer):
    """Extract possessive features efficiently using CAMeL Tools."""
    features = {}
    poss_counts = defaultdict(int)

    sentences = split_into_sentences(essay)
    sentences_with_poss = set()

    for sent_idx, sentence in enumerate(sentences):
        has_poss_in_sentence = False
        words = split_into_words(sentence)

        for word in words:
            analyses = _morph_analyzer.analyze(word)

            for analysis in analyses:
                enc0 = analysis.get('enc0', '0')

                # Look for possessive pronouns (end with _poss)
                if enc0 != '0' and enc0.endswith('_poss'):
                    has_poss_in_sentence = True
                    poss_counts['general_possessive'] += 1

                    # Extract person from the enc0 value
                    if enc0.startswith('1'):  # 1s_poss, 1p_poss
                        poss_counts['first_person_poss'] += 1
                    elif enc0.startswith('2'):  # 2ms_poss, 2fs_poss, 2p_poss
                        poss_counts['second_person_poss'] += 1
                    elif enc0.startswith('3'):  # 3ms_poss, 3fs_poss, 3p_poss
                        poss_counts['third_person_poss'] += 1

                    break  # Found possessive in this analysis

        if has_poss_in_sentence:
            sentences_with_poss.add(sent_idx)

    # Add counts to features
    for poss_type, count in poss_counts.items():
        features[f"group_{poss_type}"] = count

    # Add sentence-level statistics
    total_sentences = len(sentences)
    if total_sentences > 0:
        features["sent_count_general_possessive"] = len(sentences_with_poss)
        features["sent_percent_general_possessive"] = (len(sentences_with_poss) / total_sentences) * 100

    return features


def get_top_n_words_from_essays(essays, n=100):
    """
    Get the top N most frequent words across all essays.

    Args:
        essays (list): List of essay texts
        n (int): Number of top words to consider
    Returns:
        set: Set of top N words
    """
    # Count words across all essays
    total_word_counts = Counter()

    for essay in essays:
        # Normalize and tokenize text
        normalized_text = normalize_unicode(essay)
        words = split_into_words(normalized_text)

        # Remove stop words and normalize words
        words = [normalize_unicode(word) for word in words if word not in ARABIC_STOPWORDS]

        # Update counts
        total_word_counts.update(words)
    # Get the top N words
    top_n_words = set(word for word, count in total_word_counts.most_common(n))
    return top_n_words

def calculate_top_n_word_features(essay, total_word_counts, n=100):
    """
    Calculates features related to top N words in the essay.
    Only includes features for the pre-determined top N words across all essays.

    Args:
        essay (str): The essay text
        n (int): Number of top words to consider (default 300)
    """
    # Tokenize text
    words = split_into_words(essay)
    # Remove stop words and normalize words
    words = [normalize_unicode(word) for word in words if word not in ARABIC_STOPWORDS]
    # Count word frequencies in this essay
    word_counts = Counter(words)

    # Get sentences for sentence-level features
    sentences = split_into_sentences(essay)
    total_sentences = len(sentences)

    # Count sentences containing each word
    word_sentence_counts = defaultdict(int)
    word_sentence_percentages = defaultdict(float)
    total_word_set = set(total_word_counts)

    for sentence in sentences:
        # Normalize and tokenize sentence
        sent_words = set(normalize_unicode(w) for w in split_into_words(sentence)
                        if w not in ARABIC_STOPWORDS)

        # Count sentences containing each word
        for word in sent_words & total_word_set:
            # if word in total_word_counts:  # Only count if word is in top N
                word_sentence_counts[word] += 1

    # Calculate sentence percentages
    for word, count in word_sentence_counts.items():
        word_sentence_percentages[word] = (count / total_sentences if total_sentences > 0 else 0)

    features = {}

    # Add features only for the pre-determined top N words
    for word in total_word_counts:
        # Word count features
        features[f"top_n_word_count_{word}"] = word_counts[word]
        # Sentence count features
        features[f"top_n_num_sent_have_{word}"] = word_sentence_counts[word]
        # Sentence percentage features
        features[f"top_n_percentage_sent_have_{word}"] = word_sentence_percentages[word]

    return features

def calculate_clause_features(essay):
    """
    Calculates various clause-related features including:
    - mean_clause: average number of words in each clause
    - clause_per_s: average number of clauses per sentence
    - sent_ave_depth: average parse tree depth per sentence
    - ave_leaf_depth: average parse depth of leaf nodes
    - max_clause_in_s: maximum number of clauses in any sentence
    """
    # Common Arabic coordinating conjunctions
    all_arabic_conjunctions = [
    'و', 'أو', 'أم', 'ف', 'ثم', 'لكن', 'لكنَّ', 'بل', 'حتى', 'لا',
    'إما', 'كلا', 'إلا', 'غير', 'سوى', 'عدا', 'خلا', 'حاشا', 'ليس',
    'ما عدا', 'ما خلا', 'ما حاشا',
    # Subordinating
    'أن', 'إن', 'أنَّ', 'إنَّ', 'كأن', 'كأنَّ', 'لأن', 'كي', 'لكي',
    'إذ', 'إذا', 'لو', 'لولا', 'لوما', 'لمّا', 'مذ', 'منذ', 'ريثما',
    'بينما', 'طالما', 'كلما', 'أينما', 'حيثما', 'مهما', 'كيفما',
    'أنَّى', 'حيث', 'بحيث', 'كون', 'ولو', 'وإن',
    # Multi-word conjunctions
    'بعد أن', 'قبل أن', 'منذ أن', 'في حين', 'ما دام', 'أيًّا ما',
    'متى ما', 'إذ أن', 'على أن', 'شريطة أن', 'غير أن', 'سوى أن',
    'إلا أن', 'بيد أن', 'على الرغم من أن', 'رغم أن', 'مع أن',
    'برغم أن', 'ولو أن', 'حتى لو', 'حتى وإن'
    ]

    # Split into sentences first
    sentences = split_into_sentences(essay)

    # Initialize lists to store metrics
    clauses_per_sentence = []
    sentence_depths = []
    leaf_depths = []
    clause_lengths = []

    for sentence in sentences:

        # Create a regex pattern for splitting on conjunctions
        pattern = '|'.join(r'\s+{}\s+'.format(conj) for conj in all_arabic_conjunctions)

        # Split sentence into clauses
        clauses = re.split(pattern, sentence)
        clauses = [clause.strip() for clause in clauses if clause.strip()]

        # Store number of clauses in this sentence
        clauses_per_sentence.append(len(clauses))

        # Calculate clause lengths
        for clause in clauses:
            words = split_into_words(clause)
            clause_lengths.append(len(words))

        normalized = normalize_unicode(sentence)
        tokens = split_into_words(normalized)
        tokens = [token.strip() for token in tokens]
        # Analyze each token individually
        all_analyses = []
        for token in tokens:
            analyses = _morph_analyzer.analyze(token)
            all_analyses.extend(analyses)

        # Calculate parse tree depths
        max_depth = 0
        leaf_depth_sum = 0
        leaf_count = 0

        for analysis in all_analyses:
            # analysis is a dictionary, not a string
            if analysis and 'bw' in analysis:  # 'bw' contains morphological breakdown
                bw_analysis = analysis['bw']  # e.g., 'أُ/IV1S+وافِق/IV'

                # Count morphological complexity levels
                if bw_analysis:
                    # Count slashes and plus signs as complexity indicators
                    complexity = bw_analysis.count('/') + bw_analysis.count('+')
                    max_depth = max(max_depth, complexity)

                    # Simple leaf detection - if no subdivisions
                    if '/' not in bw_analysis and '+' not in bw_analysis:
                        leaf_depth_sum += 1
                        leaf_count += 1
                    else:
                        # Count segments
                        segments = len(bw_analysis.replace('+', '/').split('/'))
                        leaf_depth_sum += segments
                        leaf_count += 1

        sentence_depths.append(max_depth if max_depth > 0 else 1)
        if leaf_count > 0:
            leaf_depths.append(leaf_depth_sum / leaf_count)
        else:
            leaf_depths.append(1)

    # Calculate final metrics with error handling
    mean_clause = sum(clause_lengths) / len(clause_lengths) if len(clause_lengths) > 0 else 0
    clause_per_s = sum(clauses_per_sentence) / len(sentences) if len(sentences) > 0 else 0
    sent_ave_depth = sum(sentence_depths) / len(sentences) if len(sentences) > 0 else 0
    ave_leaf_depth = sum(leaf_depths) / len(leaf_depths) if len(leaf_depths) > 0 else 0
    max_clause_in_s = max(clauses_per_sentence) if clauses_per_sentence else 0

    return {
        "mean_clause": mean_clause,
        "clause_per_s": clause_per_s,
        "sent_ave_depth": sent_ave_depth,
        "ave_leaf_depth": ave_leaf_depth,
        "max_clause_in_s": max_clause_in_s
    }

def calculate_grammar_features(essay,_mle_disambiguator):
    """Calculate grammar-related features for Arabic text."""

    # Initialize feature counts
    features = {
        "auxverb": 0,           # Auxiliary verbs
        "nominalization": 0,    # Masdar forms (verbal nouns)
        "begin_w_pronoun": 0,    # Sentences starting with pronouns
        "begin_w_interrogative": 0, # Sentences starting with question words
        "begin_w_article": 0,    # Sentences starting with definite article
        "begin_w_subordination": 0, # Sentences starting with subordinating conjunctions
        "begin_w_conjunction": 0, # Sentences starting with coordinating conjunctions
        "begin_w_preposition": 0, # Sentences starting with prepositions
        "prep_comma": 0,         # Prepositions and commas
        "pronoun": 0,            # Pronouns
        "conjunction": 0,        # Conjunctions
    }

    # Lists of Arabic POS patterns to match
    aux_verbs = [
        # Future tense markers
        "سوف", "س", "سـ", "سَ",
        # Past tense markers
        "قد", "لقد", "ما زال", "ما يزال", "ما برح", "ما يبرح",
        "ما انفك", "ما ينفك", "ما فتئ", "ما يفتئ",
        # Modal auxiliaries
        "كان", "أصبح", "أضحى", "أمسى", "ظل", "بات", "صار", "ليس",
        # Aspectual auxiliaries
        "بدأ", "شرع", "أخذ", "جعل", "عاد", "رجع", "انبرى", "هب"
        ]
    conjunctions = [
        'و', 'أو', 'أم', 'ف', 'ثم', 'لكن', 'لكنَّ', 'بل', 'حتى', 'لا',
        'إما', 'كلا', 'إلا', 'غير', 'سوى', 'عدا', 'خلا', 'حاشا', 'ليس',
        'ما عدا', 'ما خلا', 'ما حاشا',
        # Subordinating
        'أن', 'إن', 'أنَّ', 'إنَّ', 'كأن', 'كأنَّ', 'لأن', 'كي', 'لكي',
        'إذ', 'إذا', 'لو', 'لولا', 'لوما', 'لمّا', 'مذ', 'منذ', 'ريثما',
        'بينما', 'طالما', 'كلما', 'أينما', 'حيثما', 'مهما', 'كيفما',
        'أنَّى', 'حيث', 'بحيث', 'كون', 'ولو', 'وإن',
        # Multi-word conjunctions
        'بعد أن', 'قبل أن', 'منذ أن', 'في حين', 'ما دام', 'أيًّا ما',
        'متى ما', 'إذ أن', 'على أن', 'شريطة أن', 'غير أن', 'سوى أن',
        'إلا أن', 'بيد أن', 'على الرغم من أن', 'رغم أن', 'مع أن',
        'برغم أن', 'ولو أن', 'حتى لو', 'حتى وإن'
        ]
    subordinating_conj = [
        # Basic subordinating conjunctions
        "أن", "إن", "أنَّ", "إنَّ", "كأن", "كأنَّ", "لأن", "كي", "لكي",
        # Time-related conjunctions
        "عندما", "بعدما", "قبلما", "حيثما", "كلما", "طالما", "متى", "إذ", "إذا",
        "حين", "حينما", "لمّا", "مذ", "منذ", "ريثما", "بينما",
        # Conditional conjunctions
        "لو", "لولا", "لوما", "إذا", "إن", "أما", "أينما", "حيثما", "مهما",
        "كيفما", "أنَّى", "حيث", "بحيث",
        # Purpose and result conjunctions
        "كي", "لكي", "حتى", "لئلا", "كي لا", "لكي لا",
        # Multi-word subordinating conjunctions
        "بعد أن", "قبل أن", "منذ أن", "في حين", "ما دام", "أيًّا ما",
        "متى ما", "إذ أن", "على أن", "شريطة أن", "غير أن", "سوى أن",
        "إلا أن", "بيد أن", "على الرغم من أن", "رغم أن", "مع أن",
        "برغم أن", "ولو أن", "حتى لو", "حتى وإن", "كون", "ولو", "وإن"
        ]
    interrogatives = [
        # Basic interrogatives
        "من", "ما", "متى", "أين", "كيف", "لماذا", "هل", "أ",
        # Additional interrogatives
        "أي", "أيان", "أنى", "كم", "كيفما", "أيما", "أينما",
        # Compound interrogatives
        "بماذا", "فيم", "علام", "إلام", "إلى متى", "من أين", "إلى أين",
        "كيف حال", "ما هو", "ما هي", "ما هم", "ما هن", "ما أنت", "ما أنتم",
        # Interrogative particles
        "أليس", "ألا", "أما", "أم", "أو", "هل", "أ",
        # Rhetorical questions
        "أترى", "أتعلم", "أتدرى", "أتعرف", "أتذكر", "أتخيل", "أتوقع",
        # Negative interrogatives
        "أليس", "ألم", "ألا", "أما", "أوما", "أفلا", "أوليس"
        ]

    # Split text into sentences
    sentences = split_into_sentences(essay)

    # Count commas
    comma_count = essay.count('،')  # Arabic comma

    # Process each sentence for sentence-beginning features
    for sentence in sentences:
        # Normalize and get all words
        normalized = normalize_unicode(sentence.strip())
        words = split_into_words(normalized)
        # Check each word in the sentence
        for i, word in enumerate(words):
            # Get POS tag for word
            word_analysis = _mle_disambiguator.disambiguate([word])
            if word_analysis and word_analysis[0].analyses:
                pos = word_analysis[0].analyses[0].analysis['pos']

                # Check patterns for each word
                if pos == 'pron' or pos == 'pron_dem':
                    if i == 0:  # Only count as beginning if it's the first word
                        features["begin_w_pronoun"] += 1
                    features["pronoun"] += 1
                elif word in interrogatives or 'interrog' in pos:
                    if i == 0:  # Only count as beginning if it's the first word
                        features["begin_w_interrogative"] += 1
                elif 'prc0' in word_analysis[0].analyses[0].analysis and word_analysis[0].analyses[0].analysis['prc0'] == 'Al_det':
                    if i == 0:  # Only count as beginning if it's the first word
                        features["begin_w_article"] += 1
                elif word in subordinating_conj or pos == 'conj_sub':
                    if i == 0:  # Only count as beginning if it's the first word
                        features["begin_w_subordination"] += 1
                elif word in conjunctions or pos == 'conj':
                    if i == 0:  # Only count as beginning if it's the first word
                        features["begin_w_conjunction"] += 1
                    features["conjunction"] += 1
                elif pos == 'prep':
                    if i == 0:  # Only count as beginning if it's the first word
                        features["begin_w_preposition"] += 1
                    features["prep_comma"] += 1
                elif pos.startswith("verb"):
                    lemma = word_analysis[0].analyses[0].analysis.get('lex', '')
                    if lemma in aux_verbs or pos == 'verb_pseudo':
                        features["auxverb"] += 1

                # Check for nominalization (masdar/verbal noun)
                if pos == 'noun' and word_analysis[0].analyses[0].analysis.get('stemcat', '').startswith('Nap'):
                    features["nominalization"] += 1

    # Add commas to prep_comma count
    features["prep_comma"] += comma_count

    return features

def analyze_dialect_usage(text, _dialect_id):
    """
    Analyze dialect usage in Arabic text using CAMeL Tools dialect identification models.

    Args:
        text (str): The Arabic text to analyze
        _dialect_id: The dialect identifier object from CAMeL Tools

    Returns:
        dict: A dictionary containing dialect usage statistics including:
            - dialect_percentages: Dictionary of dialect percentages
            - dialect_counts: Dictionary of dialect counts
            - msa_percentage: Percentage of MSA text
            - dialect_percentage: Percentage of dialectal text
            - most_common_dialect: The most frequently used dialect
    """

    # Normalize the text
    normalized_text = normalize_unicode(text)

    # Split text into sentences for better analysis
    sentences = split_into_sentences(normalized_text)

    # Initialize counters
    dialect_counts = Counter()
    total_sentences = len(sentences)

    # Analyze each sentence
    for sentence in sentences:
        # Get dialect prediction for the sentence
        prediction = _dialect_id.predict([sentence])[0]  # predict returns a list, get first element
        # Access the top dialect from the DIDPred object
        dialect = prediction.top
        dialect_counts[dialect] += 1

    # Calculate MSA vs dialect percentages
    msa_count = dialect_counts.get('MSA', 0)
    msa_percentage = (msa_count / total_sentences) * 100 if total_sentences > 0 else 0
    dialect_percentage = 100 - msa_percentage

    return {
        'dialect_counts': len(dialect_counts),
        'msa_percentage': msa_percentage,
        'dialect_percentage': dialect_percentage,
    }



def calculate_nominal_verbal_sentences(essay,_mle_disambiguator):
    sentences=split_into_sentences(essay)
    nominal_sentences=0
    verbal_sentences=0
    for sentence in sentences:
        words=split_into_words(sentence)
        # check the first three words of it contains a verb then the sentence is verbal if not then the sentence is nominal
        words_to_check=words[:3]
        for word in words_to_check:
            word_analysis = _mle_disambiguator.disambiguate([word])
            if word_analysis and word_analysis[0].analyses:
                pos = word_analysis[0].analyses[0].analysis['pos']
                if pos == 'verb':
                    verbal_sentences+=1
                    break
        else:
            nominal_sentences+=1
    return {
        'nominal_sentences': nominal_sentences,
        'verbal_sentences': verbal_sentences
    }

def count_jazm_particles(essay,_morph_analyzer):
    """
    Count jazm particles and track when they're followed by plural verbs ending with ن.
    Uses both morphological tags and particle list for validation.
    """
    jazm_stats = {
        "total_jazm": 0,
        "jazm_with_plural_verb": 0,
    }

    # Correct list of jazm particles
    jazm_particles = {
        "لم", "لما", "لام الأمر", "لا", "إن", "من", "ما", "مهما", "متى",
        "كيفما", "أنى", "أيان", "أي", "أينما", "حيثما", "إذما", "إذا",'لن'
    }

    words = split_into_words(essay)

    for i, word in enumerate(words):
        analyses = _morph_analyzer.analyze(word)
        if analyses:
            analysis = analyses[0]
            is_jazm = False
            pos_type = analysis.get('pos')
            # Check for لام using prc1 tag, and it should be followed by a verb
            # if 'prc1' in analysis and analysis['prc1'] == 'li_prep':
            if 'prc1' in analysis and pos_type == 'verb' and 'li' in analysis['prc1']:
                jazm_stats["total_jazm"] += 1 # This is for sure a jazm particle
                is_jazm = True

            # Check for negative particles using pos tag and bw field
            elif pos_type == 'part_neg':
                bw = analysis.get('bw', '')
                if '/NEG_PART' in bw:  # This will catch both لم and لن
                    is_jazm = True

            # Check for conditional and relative particles
            elif pos_type in ['part', 'conj']:
                bw = analysis.get('bw', '')
                # Check if the word contains any of the particles in the bw field
                if any(particle in bw for particle in jazm_particles):
                    is_jazm = True

            if is_jazm:
                # print(f"Found jazm particle: {word}")
                # jazm_stats["total_jazm"] += 1

                # Check if followed by a plural verb ending with ن
                if i + 1 < len(words):
                    next_word = words[i + 1]
                    next_analysis = _morph_analyzer.analyze(next_word)

                    if next_analysis and next_analysis[0].get('pos') == 'verb':
                        # jazm particles must be followed by a verb
                        if pos_type != 'verb': # This condition is to execlude لام الأمر و لام الناهية because they are already considered
                            jazm_stats["total_jazm"] += 1
                        # Check for plural verb with ن ending
                        if (next_analysis[0].get('num') == 'p' and
                            next_word.endswith('ن')):
                            jazm_stats["jazm_with_plural_verb"] += 1

    return jazm_stats

def count_conjunctions_and_transitions(essay):
    """
    Count occurrences of Arabic conjunctions and transitional phrases in the text.

    Args:
        essay (str): The Arabic text to analyze

    Returns:
        dict: A dictionary containing:
            - Counts of each conjunction and transition phrase
            - Ratio of connectives to total words
            - Ratio of unique connectives to paragraph words
            - Maximum and minimum distances between connectives
    """
    # Dictionary of conjunctions and transitions with their variations
    conjunctions_dict = {
        # Contrast and Opposition
        'الا ان': ['الا ان', 'الا أن'],
        'بيد ان': ['بيد ان', 'بيد أن'],
        'غير ان': ['غير ان', 'غير أن'],
        'على الرغم': ['على الرغم'],
        'رغمان': ['رغمان'],
        'بالرغم من': ['بالرغم من'],
        'برغم': ['برغم'],
        'بالمقابل': ['بالمقابل'],
        'في المقابل': ['في المقابل'],
        'بيد': ['بيد'],

        # Time and Sequence
        'بعدما': ['بعدما'],
        'اذ': ['اذ', 'إذ'],
        'بينما': ['بينما'],
        'عقب': ['عقب'],
        'قبيل': ['قبيل'],
        'وقبل': ['وقبل'],
        'من ثم': ['من ثم'],
        'قبل ان': ['قبل ان', 'قبل أن'],

        # Cause and Effect
        'جراء': ['جراء'],
        'نظرا ل': ['نظرا ل'],
        'بفضل': ['بفضل'],
        'لأن': ['لأن'],
        'بحيث': ['بحيث'],

        # Condition and Exception
        'الا اذا': ['الا اذا', 'الا إذا'],
        'حتى لو': ['حتى لو'],
        'لولا': ['لولا'],
        'طالما': ['طالما'],
        'كلما': ['كلما'],

        # Purpose and Comparison
        'بغية': ['بغية'],
        'كأن': ['كأن'],
        'خلافا ل': ['خلافا ل'],
        'بمعنى اخر': ['بمعنى اخر', 'بمعنى آخر'],

        # Context and Situation
        'في ظل': ['في ظل'],
        'حال': ['حال']
    }

    # Initialize results dictionary with all conjunctions set to 0
    results = {key: 0 for key in conjunctions_dict.keys()}

    # Normalize the text
    normalized_text = normalize_unicode(essay)

    # Get all words in the text
    all_words = split_into_words(normalized_text)
    total_words = len(all_words)

    # Track positions of connectives for distance calculations
    connective_positions = []
    unique_connectives = set()

    # Count occurrences of each conjunction and its variations
    for conjunction, variations in conjunctions_dict.items():
        for variation in variations:

            # Count exact matches
            # TODO: Not the best way to count variations, phrases can be rule based and word-based conjunctions should be handled by camel-tools for more accurate results
            # count = normalized_text.count(variation)
            # results[conjunction] += count
            # This pattern matches the word with optional prefixes وف or و
            # pattern = r'(?<!\w)(?:[وف]?){word}'.format(word=re.escape(variation))
            pattern = r'(?<!\w)(?:[وف]?){word}(?!\w\w)'.format(word=re.escape(variation))
            count = len(re.findall(pattern, normalized_text))
            results[conjunction] += count

            # Track positions and unique connectives
            if count > 0:
                unique_connectives.add(conjunction)
                # Find all positions of this variation
                start = 0
                while True:
                    # pos = normalized_text.find(variation, start)
                    t = normalized_text[start:]
                    pos = re.search(pattern, normalized_text[start:])
                    pos = pos.start() + start if pos else -1
                    if pos == -1:
                        break
                    # Convert character position to word position
                    word_pos = len(split_into_words(normalized_text[:pos]))
                    connective_positions.append(word_pos)
                    start = pos + 1 + len(variation)  # Move past this occurrence

    # Calculate distances between connectives
    connective_positions.sort()
    distances = []
    if len(connective_positions) > 1:
        for i in range(len(connective_positions) - 1):
            distance = connective_positions[i + 1] - connective_positions[i]
            distances.append(distance)

    # Calculate total count
    total_count = sum(results.values())
    # Calculate ratios and add to results
    if total_words > 0:
        results['connective_ratio'] = total_count / total_words
        results['unique_connective_ratio'] = len(unique_connectives) / total_words
    else:
        results['connective_ratio'] = 0
        results['unique_connective_ratio'] = 0

    # Add distance metrics
    if distances:
        results["average_connective_distance"] = sum(distances) / len(distances)
    else:
        results["average_connective_distance"] = 0


    # Add total to results
    results['total_conjunctions'] = total_count

    # Ensure all 34 conjunctions are in the results with at least 0
    for conjunction in conjunctions_dict.keys():
        if conjunction not in results:
            results[conjunction] = 0

    assert total_count == len(connective_positions), f"Total count of connectives does not match the number of positions tracked ({total_count} != {len(connective_positions)})"
    assert len(unique_connectives) <= total_count, "Unique connectives count exceeds total count"
    assert len(distances) == total_count - 1 if total_count > 1 else True, "Distance count does not match expected number of connectives"
    # all distances are less than or equal to the total number of words
    assert all(distance <= total_words for distance in distances), "Distance exceeds total number of words"

    return results

def extract_syntactic_features(essay, _mle_disambiguator):
    # Normalize the text
    normalized_text = normalize_unicode(essay)
    words = split_into_words(normalized_text)

    # Get morphological analyses
    disambiguated = _mle_disambiguator.disambiguate(words)

    # Define lemma sets for more accurate matching
    inna_lemmas = {"أن", "إن", "كأن", "لكن", "ليت", "لعل"}
    kana_lemmas = {"كان", "أضحى", "مازال", "ليس", "ماظل", "أمسى", "مافتئ", "بات", "صار", "ظل", "ماانفك", "مابرح", "مادام", "أصبح"}

    # Initialize counters
    verb_count = 0
    misspelled_count = 0
    inna_count = 0
    kana_count = 0

    spell_checker = SpellChecker(language='ar')
    # Use a set for faster lookup and only check unique words
    unique_words = set(words)
    misspelled = spell_checker.unknown(unique_words)
    misspelled_count = sum(1 for word in words if word in misspelled)

    # Count POS tags from morphological analysis
    for word in disambiguated:
        if word and len(word) > 0 and word.analyses:
            analysis = word.analyses[0].analysis
            pos = analysis.get('pos', '')
            lemma = analysis.get('lex', '')

            # Count verbs
            if pos.startswith('verb'):
                verb_count += 1

            # Count Kana words using both POS and lemma
            if pos == 'verb_pseudo' or lemma in kana_lemmas:
                kana_count += 1

            # Count Inna words using both POS and lemma
            if (pos == 'part' and lemma in inna_lemmas) or lemma in inna_lemmas:
                inna_count += 1

    features = {
        "verb_count": verb_count,
        "misspelled_count": misspelled_count,
        "inna_count": inna_count,
        "kana_count": kana_count
    }

    return features

def extract_lexical_features(essay,intro_paragraph,body_paragraph,conclusion_paragraph,_morph_analyzer):

    #Count of stop words and words without stop words
    wordsList = split_into_words(essay)
    stop_words_count =  sum(1 for word in wordsList if word in ARABIC_STOPWORDS)
    words_count_without_stopwords = sum(1 for word in wordsList if word not in ARABIC_STOPWORDS)

    #Existence of introducing and concluding words
    intro_keywords = ['نبدأ','بداية', 'نتحدث', 'نتكلم', 'نستعرض', 'الموضوع', 'في البداية', 'أولاً', 'أود أن أبدأ ب', 'أقدم', 'أعرض']
    conclusion_keywords = ['أختم','أرى', 'أخيراً', 'أرجو', 'وجهة نظر', 'أقترح', 'أتمنى', 'في الختام', 'ختاماً', 'أختاماً', 'خلاصة', 'باختصار']

    # Get robust representations of keywords using the imported function
    intro_lemmas, intro_roots = get_lemmas_and_roots(intro_keywords,_morph_analyzer)
    conclusion_lemmas, conclusion_roots = get_lemmas_and_roots(conclusion_keywords,_morph_analyzer)

    # Check introduction paragraph for intro keywords
    first_paragraph_has_intro_words = 0
    if intro_paragraph:
        # Strategy 1: Direct substring matching (dediacritized)
        dediac_intro = dediac_ar(intro_paragraph)
        for lemma in intro_lemmas:
            dediac_lemma = dediac_ar(lemma)
            if dediac_lemma in dediac_intro:
                first_paragraph_has_intro_words = 1
                break

        # Strategy 2: Morphological analysis if no direct match found
        if first_paragraph_has_intro_words == 0:
            intro_words = split_into_words(intro_paragraph)
            paragraph_lemmas = set()
            paragraph_roots = set()

            for word in intro_words:
                analyses = _morph_analyzer.analyze(word)
                if analyses:
                    lemma = analyses[0].get('lex', '')
                    if lemma:
                        paragraph_lemmas.add(dediac_ar(lemma))
                        paragraph_lemmas.add(lemma)

                    root = analyses[0].get('root', '')
                    if root:
                        paragraph_roots.add(dediac_ar(root))

            # Check lemma overlap
            if intro_lemmas & paragraph_lemmas:
                first_paragraph_has_intro_words = 1
            # Check root overlap (more generous matching)
            elif intro_roots & paragraph_roots:
                first_paragraph_has_intro_words = 1

    # Check conclusion paragraph for conclusion keywords
    last_paragraph_has_conclusion_words = 0
    if conclusion_paragraph:
        # Strategy 1: Direct substring matching (dediacritized)
        dediac_conclusion = dediac_ar(conclusion_paragraph)
        for lemma in conclusion_lemmas:
            dediac_lemma = dediac_ar(lemma)
            if dediac_lemma in dediac_conclusion:
                last_paragraph_has_conclusion_words = 1
                break

        # Strategy 2: Morphological analysis if no direct match found
        if last_paragraph_has_conclusion_words == 0:
            conclusion_words = split_into_words(conclusion_paragraph)
            paragraph_lemmas = set()
            paragraph_roots = set()

            for word in conclusion_words:
                analyses = _morph_analyzer.analyze(word)
                if analyses:
                    lemma = analyses[0].get('lex', '')
                    if lemma:
                        paragraph_lemmas.add(dediac_ar(lemma))
                        paragraph_lemmas.add(lemma)

                    root = analyses[0].get('root', '')
                    if root:
                        paragraph_roots.add(dediac_ar(root))

            # Check lemma overlap
            if conclusion_lemmas & paragraph_lemmas:
                last_paragraph_has_conclusion_words = 1
            # Check root overlap (more generous matching)
            elif conclusion_roots & paragraph_roots:
                last_paragraph_has_conclusion_words = 1

    features = {
        "stop_words_count": stop_words_count,
        "words_count_without_stopwords": words_count_without_stopwords,
        "first_paragraph_has_intro_words": first_paragraph_has_intro_words,
        "last_paragraph_has_conclusion_words": last_paragraph_has_conclusion_words
    }
    return features
''', encoding="utf-8")
print("Wrote syntactic_features.py")

Wrote syntactic_features.py


In [11]:
from pathlib import Path
Path("/content/aes_project/extractor_script.py").write_text(r'''from camel_tools_init import (get_disambiguator, get_analyzer, get_sentiment_analyzer,
                              get_bert_model, get_tagger, _mle_disambiguator,
                              _morph_analyzer, _sentiment_analyzer, _bert_tokenizer,
                              _bert_model, _default_tagger,get_dialect_id,_dialect_id)
from essay_proccessing import split_into_sentences, split_into_paragraphs
from pos_features import get_top_n_pos_tags,get_top_n_pos_bigrams,get_essay_pos_features
from readability_measures import calculate_readability_scores
from semantic_features import calculate_prompt_adherence_features,calculate_sentiment_scores,calculate_semantic_similarities,calculate_sent_match_words
from surface_level_features import (calculate_religious_phrases,
    calculate_advanced_punctuation_features,extract_surface_features,
    calculate_lemma_features,calculate_variance_features,long_words_count,
    calculate_punctuation_counts,calculate_dup_punctuation_count,long_words_count)
from syntactic_features import (count_jazm_particles,analyze_dialect_usage,
    calculate_syllable_features,calculate_pronoun_features,
    calculate_possessive_features,calculate_grammar_features,
    calculate_nominal_verbal_sentences,count_conjunctions_and_transitions,
    extract_syntactic_features,extract_lexical_features,
    get_top_n_words_from_essays,calculate_top_n_word_features)
import pandas as pd
from camel_tools.utils.normalize import normalize_unicode
import numpy as np
import os
from tqdm import tqdm
# contansts
INPUT_FILE_PATH='/content/all_prompts_all_folds_combined.csv'
INPUT_PARAGRAPHS_FILE_PATH='../../../../shared/Arabic_Dataset/cq_essay_paragraphs.csv'
OUTPUT_FILE_PATH='/content/output_features/laila_features.csv'
OUTPUT_DIR='/content/output_features'
PROMPTS_FILE_PATH='../../../../shared/Arabic_Dataset/arabic_prompts'
# New JSON prompts path (local path without sftp scheme)
PROMPTS_JSON_PATH='/content/all_prompts.json'
PROMPT_ID=None  # ضع رقمًا من 1 إلى 8 إذا أردت موضوعًا واحدًا فقط

import json

def load_prompts():
    """Load prompts from a single JSON file and return mapping by prompt_id.

    Returns a dict:
        { prompt_id: { 'text': str, 'type': str, 'type_encoded': int } }
    """
    with open(PROMPTS_JSON_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)

    type_to_code = {
        'persuasive': 1,
        'explanatory': 2,
    }

    prompts = {}
    for item in data:
        prompt_id = item.get('prompt_id')
        prompt_text = item.get('prompt_text', '')
        prompt_type = item.get('prompt_type', '').strip().lower()
        prompts[prompt_id] = {
            'text': prompt_text,
            'type': prompt_type,
            'type_encoded': type_to_code.get(prompt_type, 0),
        }

    return prompts

def main():
    # Initialize models
    _mle_disambiguator = get_disambiguator()
    _morph_analyzer = get_analyzer()
    _sentiment_analyzer = get_sentiment_analyzer()
    _bert_tokenizer, _bert_model = get_bert_model()
    _default_tagger = get_tagger()
    _dialect_id=get_dialect_id()
    df=pd.read_csv(INPUT_FILE_PATH)
    if PROMPT_ID is not None:
        df = df[df['prompt_id'] == PROMPT_ID].copy()
        print(f"Loaded {len(df)} essays for prompt_id={PROMPT_ID}")
    else:
        print(f"Loaded {len(df)} essays (all prompts)")
    # paragraphs_df=pd.read_csv(INPUT_PARAGRAPHS_FILE_PATH)  # Old way: read pre-split paragraphs
    top_n_pos_tags=get_top_n_pos_tags(df['essay'],_mle_disambiguator,100)
    top_n_pos_bigrams=get_top_n_pos_bigrams(df['essay'],_mle_disambiguator,100)
    total_word_counts=get_top_n_words_from_essays(df['essay'],100)
    # clause_analyzer=ClauseAnalyzer()
    prompts=load_prompts()
    # Read the input file
    features_df=pd.DataFrame()

    # Add progress bar to the loop
    print(f"Processing {len(df)} essays...")
    for index, row in tqdm(df.iterrows(), total=len(df), desc="Extracting features", unit="essay"):
        essay=row['essay']
        id=row['essay_id']
        prompt_info = prompts.get(row['prompt_id'], {'text': '', 'type': '', 'type_encoded': 0})
        prompt = prompt_info['text']
        prompt_type_encoded = prompt_info['type_encoded']
        # Old way: use paragraphs_df to get intro/body/conclusion
        # intro_paragraph=paragraphs_df[paragraphs_df['essay_id']==id]['introduction'].values[0]
        # body_paragraph=paragraphs_df[paragraphs_df['essay_id']==id]['body'].values[0]
        # conclusion_paragraph=paragraphs_df[paragraphs_df['essay_id']==id]['conclusion'].values[0]
        # n_paragraphs = 3
        # if intro_paragraph is np.nan:
        #     intro_paragraph=''
        #     n_paragraphs -= 1
        # if body_paragraph is np.nan:
        #     body_paragraph=''
        #     n_paragraphs -= 1
        # if conclusion_paragraph is np.nan:
        #     conclusion_paragraph=''
        #     n_paragraphs -= 1

        # New way: split paragraphs by newline characters from the essay text
        raw_paragraphs = [p for p in normalize_unicode(essay).split('\n')]
        paragraphs = [p.strip() for p in raw_paragraphs if p.strip() != '']
        n_paragraphs = len(paragraphs)
        if n_paragraphs == 0:
            intro_paragraph = ''
            body_paragraph = ''
            conclusion_paragraph = ''
        elif n_paragraphs == 1:
            intro_paragraph = paragraphs[0]
            body_paragraph = ''
            conclusion_paragraph = ''
        elif n_paragraphs == 2:
            intro_paragraph = paragraphs[0]
            body_paragraph = ''
            conclusion_paragraph = paragraphs[1]
        else:
            # For 3+ paragraphs: first -> intro, last -> conclusion, middle -> body
            intro_paragraph = paragraphs[0]
            conclusion_paragraph = paragraphs[-1]
            body_paragraph = '\n'.join(paragraphs[1:-1])
        longest_paragaph_length=max(len(intro_paragraph),len(body_paragraph),len(conclusion_paragraph))
        shortest_paragaph_length=min(len(intro_paragraph),len(body_paragraph),len(conclusion_paragraph))
        # Surface level features
        surface_features=extract_surface_features(essay,intro_paragraph,body_paragraph,conclusion_paragraph)
        religious_features=calculate_religious_phrases(intro_paragraph,body_paragraph,conclusion_paragraph)
        lemma_features=calculate_lemma_features(essay,_morph_analyzer)
        variance_features=calculate_variance_features(essay)
        long_words_features={"long_words_count": long_words_count(essay)}
        punctuation_counts_features=calculate_punctuation_counts(essay,_mle_disambiguator)
        dup_punctuation_features={"dup_punctuation_count": calculate_dup_punctuation_count(essay,_mle_disambiguator)}
        advanced_punctuation_features=calculate_advanced_punctuation_features(essay,_mle_disambiguator)

        # POS features
        pos_features=get_essay_pos_features(essay,top_n_pos_tags,top_n_pos_bigrams,_mle_disambiguator,100)
        # Readability features
        readability_features=calculate_readability_scores(essay,_mle_disambiguator)
        # Semantic features
        semantic_features=calculate_semantic_similarities(intro_paragraph,body_paragraph,conclusion_paragraph,_bert_tokenizer, _bert_model)
        sentiment_features=calculate_sentiment_scores(essay)
        sent_match_words_features=calculate_sent_match_words(essay)
        prompt_adherence_features=calculate_prompt_adherence_features(essay,prompt,_bert_tokenizer, _bert_model)
        # syntax features
        dialect_features=analyze_dialect_usage(essay,_dialect_id)
        syllable_features=calculate_syllable_features(essay,_mle_disambiguator)
        pronoun_features=calculate_pronoun_features(essay,_mle_disambiguator)
        possessive_features=calculate_possessive_features(essay,_morph_analyzer)
        # clause_features=clause_analyzer.calculate_features(essay)
        grammar_features=calculate_grammar_features(essay,_mle_disambiguator)
        nominal_verbal_sentences=calculate_nominal_verbal_sentences(essay,_mle_disambiguator)
        conjunctions_and_transitions=count_conjunctions_and_transitions(essay)
        syntactic_features=extract_syntactic_features(essay,_mle_disambiguator)
        lexical_features=extract_lexical_features(essay,intro_paragraph,body_paragraph,conclusion_paragraph,_morph_analyzer)
        top_n_word_features=calculate_top_n_word_features(essay,total_word_counts,100)
        jazm_features=count_jazm_particles(essay,_morph_analyzer)

        # paragraph features
        intro_paragraph_features=extract_surface_features(intro_paragraph,"","","")
        # add intro paragraph to the key of every feature
        intro_paragraph_features={f"intro_{k}": v for k, v in intro_paragraph_features.items()}
        body_paragraph_features=extract_surface_features(body_paragraph,"","","")
        body_paragraph_features={f"body_{k}": v for k, v in body_paragraph_features.items()}
        conclusion_paragraph_features=extract_surface_features(conclusion_paragraph,"","","")
        conclusion_paragraph_features={f"conclusion_{k}": v for k, v in conclusion_paragraph_features.items()}
        intro_paragraph_punctuation_features=calculate_punctuation_counts(intro_paragraph,_mle_disambiguator)
        intro_paragraph_punctuation_features={f"intro_{k}": v for k, v in intro_paragraph_punctuation_features.items()}
        body_paragraph_punctuation_features=calculate_punctuation_counts(body_paragraph,_mle_disambiguator)
        body_paragraph_punctuation_features={f"body_{k}": v for k, v in body_paragraph_punctuation_features.items()}
        conclusion_paragraph_punctuation_features=calculate_punctuation_counts(conclusion_paragraph,_mle_disambiguator)
        conclusion_paragraph_punctuation_features={f"conclusion_{k}": v for k, v in conclusion_paragraph_punctuation_features.items()}



        features_df=pd.concat([features_df,pd.DataFrame({
            'essay_id':id,
            'prompt_id':row['prompt_id'],
            'essay':essay,
            'prompt':prompt,
            'prompt_type':prompt_info['type'],
            'prompt_type_encoded':prompt_type_encoded,
            'relevance':row['relevance'],
            'organization':row['organization'],
            'vocabulary':row['vocabulary'],
            'style':row['style'],
            'development':row['development'],
            'mechanics':row['mechanics'],
            'grammar':row['grammar'],
            'holistic':row['holistic'],
            'longest_paragraph_length':longest_paragaph_length,
            'shortest_paragraph_length':shortest_paragaph_length,
            'longest_paragraph_length_ratio':longest_paragaph_length/shortest_paragaph_length if shortest_paragaph_length!=0 else 0,
            'shortest_paragraph_length_ratio':shortest_paragaph_length/longest_paragaph_length if longest_paragaph_length!=0 else 0,
            'no_of_words_in_first':len(normalize_unicode(intro_paragraph).split()),
            'no_of_words_in_body':len(normalize_unicode(body_paragraph).split()),
            'no_of_words_in_conclusion':len(normalize_unicode(conclusion_paragraph).split()),
            'more_than_1_paragraph': 1 if n_paragraphs > 1 else 0,
            'colon_exists': 1 if ':' in essay else 0,
            'paranthesis_exists': 1 if '(' in essay else 0,
            'question_mark_exists': 1 if ('?' in essay or '؟' in essay) else 0,
            **pos_features,
            **readability_features,
            **semantic_features,
            **sentiment_features,
            **sent_match_words_features,
            **prompt_adherence_features,
            **dialect_features,
            **surface_features,
            **religious_features,
            **lemma_features,
            **variance_features,
            **long_words_features,
            **punctuation_counts_features,
            **dup_punctuation_features,
            **advanced_punctuation_features,
            **syllable_features,
            **pronoun_features,
            **possessive_features,
            # **clause_features,
            **grammar_features,
            **nominal_verbal_sentences,
            **conjunctions_and_transitions,
            **syntactic_features,
            **lexical_features,
            **top_n_word_features,
            **jazm_features,
            **intro_paragraph_features,
            **body_paragraph_features,
            **conclusion_paragraph_features,
            **intro_paragraph_punctuation_features,
            **body_paragraph_punctuation_features,
            **conclusion_paragraph_punctuation_features,
        },index=[0])], ignore_index=True)

    print("\nFeature extraction completed!")
    # print stats of features
    print(features_df.describe())
    # create the dir if not exists
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
    # save features to csv
    features_df.to_csv(OUTPUT_FILE_PATH,index=False)
    print(f"Features saved to: {OUTPUT_FILE_PATH}")

main()


''', encoding="utf-8")
print("Wrote extractor_script.py")

Wrote extractor_script.py


In [12]:
# عدّل هنا فقط إذا أردت موضوعًا واحدًا
PROMPT_ID = 1   # مثال: 1 أو 2 أو 8

import re
from pathlib import Path

# نحدّث المتغير داخل extractor_script.py
p = Path("/content/aes_project/extractor_script.py")
txt = p.read_text(encoding="utf-8")

txt = re.sub(
    r"PROMPT_ID=None\s+# ضع رقمًا من 1 إلى 8 إذا أردت موضوعًا واحدًا فقط",
    f"PROMPT_ID={repr(PROMPT_ID)}  # ضع رقمًا من 1 إلى 8 إذا أردت موضوعًا واحدًا فقط",
    txt
)

p.write_text(txt, encoding="utf-8")
print("PROMPT_ID set to", PROMPT_ID)

PROMPT_ID set to 1


In [13]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [14]:
%cd /content/aes_project
!mkdir -p /content/output_features
!python extractor_script.py 2>&1 | tee /content/extract_log.txt

/content/aes_project
2026-04-18 18:48:24.227077: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-18 18:48:24.235213: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776538104.244099    4081 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776538104.247295    4081 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776538104.255424    4081 computation_placer.cc:177] computation placer already registered. Please check linka

In [15]:
import os
import pandas as pd

# ====================================================
# البحث عن الملف الناتج
# ====================================================
raw_path = "/content/output_features/laila_features.csv"

if not os.path.exists(raw_path):
    print("❌ الملف الناتج غير موجود. راجع سجل التشغيل أعلاه.")
    try:
        log = open('/content/extract_log.txt', encoding='utf-8', errors='ignore').read()
        print("آخر 5000 حرف من السجل:\n", log[-5000:])
    except FileNotFoundError:
        print("لا يوجد ملف لوج.")
else:
    df_raw = pd.read_csv(raw_path)
    print(f"✅ الملف الخام: {df_raw.shape[0]} مقالة × {df_raw.shape[1]} عمود")
    print(f"\n🔍 عدد الأعمدة الفارغة: {(df_raw.isnull().sum() > 0).sum()}")
    print(f"🔍 مجموع القيم الفارغة: {df_raw.isnull().sum().sum()}")
    df_raw.head(2)

✅ الملف الخام: 1122 مقالة × 819 عمود

🔍 عدد الأعمدة الفارغة: 130
🔍 مجموع القيم الفارغة: 120720


In [16]:
import pandas as pd
import numpy as np

# ====================================================
# 📂 قراءة الملف الخام
# ====================================================
RAW_PATH = "/content/output_features/laila_features.csv"
df = pd.read_csv(RAW_PATH)
print(f"📊 البداية: {df.shape[0]} مقالة × {df.shape[1]} عمود\n")

# الأعمدة التي لا نلمسها أبداً (ميتاداتا + درجات)
META_COLS = ['essay_id','prompt_id','essay','prompt','prompt_type','prompt_type_encoded',
             'relevance','organization','vocabulary','style','development','mechanics',
             'grammar','holistic']

# سجل الأعمدة المحذوفة (للتقرير)
removed_report = {
    'topic_words_top_n': [],
    'sparse_pronoun_individual': [],
    'constant_columns': [],
    'duplicate_columns': [],
    'filled_with_zero': [],
}

# ====================================================
# 1️⃣ حذف فيتشرز كلمات الموضوع (top_n_*)
# ====================================================
# أعمدة مثل top_n_word_count_التواصل, top_n_percentage_sent_have_التعلم
# هذه كلمات مربوطة بموضوع محدد (للـ prompt الحالي فقط)
# ❌ لا نريد النموذج يحفظ كلمات الموضوع بل يقيس جودة الكتابة
top_n_cols = [c for c in df.columns if c.startswith('top_n_')]
df.drop(columns=top_n_cols, inplace=True)
removed_report['topic_words_top_n'] = top_n_cols
print(f"1️⃣  حذف فيتشرز كلمات الموضوع (top_n_*): {len(top_n_cols)} عمود")

# ====================================================
# 2️⃣ حذف فيتشرز الضمائر الفردية الـ sparse
# ====================================================
# أعمدة مثل pron_pron_هي, pron_pron_أنتم
# هذه الأعمدة null في أكثر من 50% من المقالات (بعضها 99%!)
null_counts = df.isnull().sum()
null_cols = null_counts[null_counts > 0].index.tolist()

sparse_pronoun_cols = [c for c in null_cols
                       if c.startswith('pron_pron_')
                       or c.startswith('sent_count_pron_')
                       or c.startswith('sent_percent_pron_')]

df.drop(columns=sparse_pronoun_cols, inplace=True)
removed_report['sparse_pronoun_individual'] = sparse_pronoun_cols
print(f"2️⃣  حذف فيتشرز ضمائر فردية sparse: {len(sparse_pronoun_cols)} عمود")

# ====================================================
# 3️⃣ ملء فيتشرز مجموعات الضمائر بـ 0
# ====================================================
# الأعمدة الملخصة (group_demonstrative إلخ) null تعني المجموعة لم تظهر = 0
null_counts2 = df.isnull().sum()
to_fill_cols = null_counts2[null_counts2 > 0].index.tolist()
df[to_fill_cols] = df[to_fill_cols].fillna(0)
removed_report['filled_with_zero'] = to_fill_cols
print(f"3️⃣  ملء مجموعات الضمائر بـ 0: {len(to_fill_cols)} عمود")

# ====================================================
# 4️⃣ حذف الأعمدة الثابتة (زيرو-فاريانس)
# ====================================================
# أعمدة قيمتها ثابتة (كلها صفر أو كلها نفس القيمة) لا تفيد النموذج أبداً
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
feature_numeric = [c for c in numeric_cols if c not in META_COLS]

constant_cols = [c for c in feature_numeric if df[c].nunique() <= 1]
df.drop(columns=constant_cols, inplace=True)
removed_report['constant_columns'] = constant_cols
print(f"4️⃣  حذف أعمدة ثابتة (كلها صفر): {len(constant_cols)} عمود")

# ====================================================
# 5️⃣ حذف الأعمدة المكررة
# ====================================================
# أعمدة قيمها مطابقة تماماً لأعمدة أخرى ← حذف التكرارات
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
feature_numeric = [c for c in numeric_cols if c not in META_COLS]

seen = {}
duplicate_cols = []
for col in feature_numeric:
    signature = tuple(df[col].round(6).tolist())
    if signature in seen:
        duplicate_cols.append((col, seen[signature]))
    else:
        seen[signature] = col

dup_to_drop = [dup for dup, orig in duplicate_cols]
df.drop(columns=dup_to_drop, inplace=True)
removed_report['duplicate_columns'] = duplicate_cols
print(f"5️⃣  حذف أعمدة مكررة: {len(dup_to_drop)} عمود")

# ====================================================
# 📊 الناتج النهائي
# ====================================================
total_dropped = (len(top_n_cols) + len(sparse_pronoun_cols) +
                 len(constant_cols) + len(dup_to_drop))
print(f"\n🎯 النهائي: {df.shape[0]} مقالة × {df.shape[1]} عمود")
print(f"✅ إجمالي الأعمدة المحذوفة: {total_dropped}")
print(f"✅ الأعمدة الممتلئة بـ 0: {len(to_fill_cols)}")
print(f"✅ القيم الفارغة المتبقية: {df.isnull().sum().sum()}")

df.head(2)

📊 البداية: 1122 مقالة × 819 عمود

1️⃣  حذف فيتشرز كلمات الموضوع (top_n_*): 300 عمود
2️⃣  حذف فيتشرز ضمائر فردية sparse: 108 عمود
3️⃣  ملء مجموعات الضمائر بـ 0: 22 عمود
4️⃣  حذف أعمدة ثابتة (كلها صفر): 14 عمود
5️⃣  حذف أعمدة مكررة: 3 عمود

🎯 النهائي: 1122 مقالة × 394 عمود
✅ إجمالي الأعمدة المحذوفة: 425
✅ الأعمدة الممتلئة بـ 0: 22
✅ القيم الفارغة المتبقية: 0


,essay_id,prompt_id,essay,prompt,prompt_type,prompt_type_encoded,relevance,organization,vocabulary,style,...,sent_percent_third_person,group_second_person_poss,group_first_person,sent_count_first_person,sent_percent_first_person,group_second_person,sent_count_second_person,sent_percent_second_person,sent_count_interrogative,sent_percent_interrogative
0,10825,1,قربا مربط النعامة مني طال ليلي على الليالي الط...,باتَ اِهْتمام وحماس المراهقين لِتعلُّم رِياضةٍ...,explanatory,2,0,0,0,0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,11087,1,أنتشرت فالآونه الاخيرة مشكلة كبيرة في رأيي، وه...,باتَ اِهْتمام وحماس المراهقين لِتعلُّم رِياضةٍ...,explanatory,2,2,4,4,4,...,17.647059,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
# ====================================================
# حفظ تقرير بما تم حذفه/تعديله
# ====================================================
report_path = "/content/output_features/removed_columns_report.txt"

with open(report_path, 'w', encoding='utf-8') as f:
    f.write("=" * 70 + "\n")
    f.write("تقرير الأعمدة المحذوفة والمُعدَّلة\n")
    f.write("=" * 70 + "\n\n")

    f.write(f"📊 الأبعاد: من 819 عمود خام → {df.shape[1]} عمود نظيف\n\n")

    # 1. Topic words dropped
    f.write(f"1️⃣ فيتشرز كلمات الموضوع top_n_* (محذوفة - مربوطة بموضوع محدد): {len(removed_report['topic_words_top_n'])}\n")
    f.write("-" * 70 + "\n")
    for c in removed_report['topic_words_top_n']:
        f.write(f"   ❌ {c}\n")

    f.write(f"\n2️⃣ فيتشرز ضمائر فردية sparse (محذوفة - null > 50%): {len(removed_report['sparse_pronoun_individual'])}\n")
    f.write("-" * 70 + "\n")
    for c in removed_report['sparse_pronoun_individual']:
        f.write(f"   ❌ {c}\n")

    f.write(f"\n3️⃣ أعمدة مجموعات ضمائر ممتلئة بـ 0 (null = لم يظهر): {len(removed_report['filled_with_zero'])}\n")
    f.write("-" * 70 + "\n")
    for c in removed_report['filled_with_zero']:
        f.write(f"   ✏️  {c}\n")

    f.write(f"\n4️⃣ أعمدة ثابتة (محذوفة - قيمة واحدة لكل المقالات): {len(removed_report['constant_columns'])}\n")
    f.write("-" * 70 + "\n")
    for c in removed_report['constant_columns']:
        f.write(f"   ❌ {c}\n")

    f.write(f"\n5️⃣ أعمدة مكررة (محذوفة - نفس قيم عمود آخر): {len(removed_report['duplicate_columns'])}\n")
    f.write("-" * 70 + "\n")
    for dup, orig in removed_report['duplicate_columns']:
        f.write(f"   ❌ {dup}  ≡  {orig}\n")

print(f"✅ تم حفظ التقرير في: {report_path}")
print(f"\n--- ملخص ---")
print(f"كلمات الموضوع المحذوفة: {len(removed_report['topic_words_top_n'])}")
print(f"ضمائر sparse محذوفة: {len(removed_report['sparse_pronoun_individual'])}")
print(f"أعمدة ثابتة محذوفة: {len(removed_report['constant_columns'])}")
print(f"أعمدة مكررة محذوفة: {len(removed_report['duplicate_columns'])}")
print(f"أعمدة ممتلئة بـ 0: {len(removed_report['filled_with_zero'])}")

✅ تم حفظ التقرير في: /content/output_features/removed_columns_report.txt

--- ملخص ---
كلمات الموضوع المحذوفة: 300
ضمائر sparse محذوفة: 108
أعمدة ثابتة محذوفة: 14
أعمدة مكررة محذوفة: 3
أعمدة ممتلئة بـ 0: 22


In [18]:
import os
import shutil
from google.colab import files

OUTPUT_DIR = "/content/output_features"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ====================================================
# 1. حفظ الملف النظيف
# ====================================================
clean_path = f"{OUTPUT_DIR}/laila_features_clean.csv"
df.to_csv(clean_path, index=False)
print(f"✅ الملف النظيف: {clean_path}")
print(f"   الأبعاد: {df.shape[0]} مقالة × {df.shape[1]} عمود")

# ====================================================
# 2. إعادة تسمية الملف الخام للحفظ (إن وُجد)
# ====================================================
raw_path_orig = f"{OUTPUT_DIR}/laila_features.csv"
raw_path_final = f"{OUTPUT_DIR}/laila_features_raw.csv"
if os.path.exists(raw_path_orig):
    shutil.copy(raw_path_orig, raw_path_final)
    print(f"✅ الملف الخام: {raw_path_final}")

# ====================================================
# 3. تنزيل الملفات
# ====================================================
print("\n📥 بدء التنزيل...")
files.download(clean_path)
if os.path.exists(raw_path_final):
    files.download(raw_path_final)
report_path = f"{OUTPUT_DIR}/removed_columns_report.txt"
if os.path.exists(report_path):
    files.download(report_path)
print("✅ تم!")

✅ الملف النظيف: /content/output_features/laila_features_clean.csv
   الأبعاد: 1122 مقالة × 394 عمود
✅ الملف الخام: /content/output_features/laila_features_raw.csv

📥 بدء التنزيل...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ تم!
